# Exploratory Data Analysis (EDA)
## Amazon ML Challenge 2026 — Business Entity Resolution

Structured exploratory analysis of challenge data across Source 1, 2, 3, and Ground Truth.

In [1]:
import pandas as pd
import numpy as np
from rapidfuzz.fuzz import ratio
from tqdm import tqdm

path = "../ML_dataset/student_resource/dataset/train"

s1 = pd.read_csv(f"{path}/train_source1.tsv", sep="\t")
s2 = pd.read_csv(f"{path}/train_source2.tsv", sep="\t")
s3 = pd.read_csv(f"{path}/train_source3.tsv", sep="\t")
source1 = s1
gt = pd.read_csv(f"{path}/train_ground_truth.tsv", sep="\t")
source2 = pd.read_csv(f"{path}/train_source2.tsv", sep="\t")
source3 = pd.read_csv(f"{path}/train_source3.tsv", sep="\t")

In [3]:
datasets = {
    "Source 1": s1,
    "Source 2": s2,
    "Source 3": s3,
    "Ground Truth": gt
}

for name, df in datasets.items():
    print(f"\n{name}")
    print("Shape:", df.shape)
    print("Columns:", list(df.columns))
    print(df.head(3))


Source 1
Shape: (2206821, 4)
Columns: ['entity_id', 'business_name', 'business_address', 'country']
      entity_id        business_name                        business_address  \
0  S1-925783039  Orelee's Barbershop  1795 Westchester Drive, High Point, NC   
1  S1-773889195          Prime Money         17560 Ellis Road, Tahlequah, OK   
2  S1-377745466        B+ Retail Inc     1712 Montebello Avenue, Phoenix, AZ   

  country  
0      US  
1      US  
2      US  

Source 2
Shape: (5034616, 4)
Columns: ['entity_id', 'business_name', 'business_address', 'country']
      entity_id                    business_name  \
0  S2-166376419  राम मार्केटिंग प्राइवेट लिमिटेड   
1  S2-764573417     -- Holloway Peak Inc Seafood   
2  S2-639257739         आदित्य प्रॉपर्टीज एलएलपी   

                                   business_address country  
0      KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi   India  
1                         105 ELM ST, MORGANTON, NC      US  
2  G-3/571, GULMOHAR COLONY, BHOPA

In [9]:
for name, df in datasets.items():
    print(f"\n{name}")
    print(df.isna().sum())
    print(df.isna().mean().mul(100).round(2))


Source 1
entity_id           0
business_name       0
business_address    0
country             0
dtype: int64
entity_id           0.0
business_name       0.0
business_address    0.0
country             0.0
dtype: float64

Source 2
entity_id                0
business_name            2
business_address    168967
country                  0
dtype: int64
entity_id           0.00
business_name       0.00
business_address    3.36
country             0.00
dtype: float64

Source 3
entity_id                0
business_name           13
business_address    175916
country                  0
dtype: int64
entity_id           0.00
business_name       0.00
business_address    3.33
country             0.00
dtype: float64

Ground Truth
source1_entity_id          0
matched_entity_ids    123247
dtype: int64
source1_entity_id     0.00
matched_entity_ids    5.58
dtype: float64


In [11]:
for name, df in {
    "Source 1": s1,
    "Source 2": s2,
    "Source 3": s3
}.items():
    print(f"\n{name}")
    print("Duplicate rows:", df.duplicated().sum())
    print("Duplicate entity IDs:", df["entity_id"].duplicated().sum())

print("\nGround Truth")
print("Duplicate rows:", gt.duplicated().sum())
print("Duplicate Source-1 IDs:", gt["source1_entity_id"].duplicated().sum())


Source 1
Duplicate rows: 0
Duplicate entity IDs: 0

Source 2
Duplicate rows: 0
Duplicate entity IDs: 0

Source 3
Duplicate rows: 0
Duplicate entity IDs: 0

Ground Truth
Duplicate rows: 0
Duplicate Source-1 IDs: 0


In [12]:
for name, df in {"S1": s1, "S2": s2, "S3": s3}.items():
    print(f"\n{name}")
    print("Duplicate rows:", df.duplicated().sum())
    print("Duplicate IDs:", df["entity_id"].duplicated().sum())
    
    print("Duplicate names:", df["business_name"].duplicated().sum())


S1
Duplicate rows: 0
Duplicate IDs: 0
Duplicate names: 667592

S2
Duplicate rows: 0
Duplicate IDs: 0
Duplicate names: 632607

S3
Duplicate rows: 0
Duplicate IDs: 0
Duplicate names: 633994


In [13]:
for name, df in {"S1": s1, "S2": s2, "S3": s3}.items():
    counts = df["business_name"].value_counts()
    print(f"{name}:")
    print("Unique names:", counts.size)
    print("Names appearing >1 time:", (counts > 1).sum())
    print("Most common names:")
    print(counts.head(5))
    print()

S1:
Unique names: 1539229
Names appearing >1 time: 177793
Most common names:
business_name
Primary Care Group         253
Ear Nose & Throat Group    251
Pediatric Group            222
Womens Health Group        220
Physical Therapy Group     218
Name: count, dtype: int64

S2:
Unique names: 4402008
Names appearing >1 time: 239778
Most common names:
business_name
Primary Care         320
Physical Therapy     307
Urgent Care          297
Womens Health        297
Behavioral Health    296
Name: count, dtype: int64

S3:
Unique names: 4651608
Names appearing >1 time: 258275
Most common names:
business_name
Primary Care        421
Physical Therapy    399
Pediatric Dental    393
Womens Health       379
Urgent Care         377
Name: count, dtype: int64



In [14]:
for name, df in {"S1": s1, "S2": s2, "S3": s3}.items():
    print(f"\n{name}")
    print(df["country"].value_counts().head(20))


S1
country
US       1323633
India     883188
Name: count, dtype: int64

S2
country
US       3016817
India    2017799
Name: count, dtype: int64

S3
country
US       3170056
India    2115547
Name: count, dtype: int64


In [15]:
for name, df in {"S1": s1, "S2": s2, "S3": s3}.items():
    length = df["business_address"].fillna("").str.len()

    print(f"\n{name}")
    print(length.describe())


S1
count    2.206821e+06
mean     5.206621e+01
std      2.532972e+01
min      1.100000e+01
25%      3.300000e+01
50%      4.100000e+01
75%      7.000000e+01
max      2.560000e+02
Name: business_address, dtype: float64

S2
count    5.034616e+06
mean     4.622591e+01
std      2.484467e+01
min      0.000000e+00
25%      3.000000e+01
50%      3.700000e+01
75%      6.100000e+01
max      2.490000e+02
Name: business_address, dtype: float64

S3
count    5.285603e+06
mean     4.671444e+01
std      2.164981e+01
min      0.000000e+00
25%      3.500000e+01
50%      4.200000e+01
75%      5.400000e+01
max      2.400000e+02
Name: business_address, dtype: float64


In [16]:
for name, df in {"S1": s1, "S2": s2, "S3": s3}.items():
    length = df["business_name"].fillna("").str.len()

    print(f"\n{name}")
    print(length.describe())


S1
count    2.206821e+06
mean     2.403440e+01
std      7.740533e+00
min      3.000000e+00
25%      1.800000e+01
50%      2.400000e+01
75%      3.000000e+01
max      1.050000e+02
Name: business_name, dtype: float64

S2
count    5.034616e+06
mean     2.510357e+01
std      8.894694e+00
min      0.000000e+00
25%      1.900000e+01
50%      2.500000e+01
75%      3.100000e+01
max      1.040000e+02
Name: business_name, dtype: float64

S3
count    5.285603e+06
mean     2.520214e+01
std      9.493635e+00
min      0.000000e+00
25%      1.800000e+01
50%      2.500000e+01
75%      3.100000e+01
max      1.230000e+02
Name: business_name, dtype: float64


In [17]:
matches = gt["matched_entity_ids"].fillna("")

match_count = matches.apply(
    lambda x: 0 if x == "" else len(x.split(","))
)

match_count.describe()

count    2.206821e+06
mean     3.461253e+00
std      1.705323e+00
min      0.000000e+00
25%      2.000000e+00
50%      3.000000e+00
75%      5.000000e+00
max      1.100000e+01
Name: matched_entity_ids, dtype: float64

In [18]:
match_count.value_counts().sort_index()

matched_entity_ids
0     123247
1     119157
2     375212
3     530841
4     484115
5     321957
6     164868
7      63968
8      18680
9       4205
10       534
11        37
Name: count, dtype: int64

In [21]:
print("Jupyter is working")

Jupyter is working


In [22]:
gt["matched_entity_ids"].head(10)

0    S2-681193310,S2-743505751,S3-775321672,S3-1129...
1    S2-249013014,S2-197070651,S3-478195123,S3-3843...
2               S2-790675320,S2-479876582,S3-878454467
3                S2-153058913,S2-24659151,S3-679606215
4    S2-478959098,S2-553508714,S2-625774905,S3-7280...
5    S2-755677256,S3-187831601,S3-641489370,S3-4762...
6                            S2-660036492,S3-804600254
7                            S3-274817120,S3-312496301
8                            S2-648035184,S3-588502663
9                S2-383871912,S3-74481402,S3-576451439
Name: matched_entity_ids, dtype: object

In [23]:
print(gt["source1_entity_id"].isin(s1["entity_id"]).mean())

1.0


In [24]:
print(gt["source1_entity_id"].isin(s1["entity_id"]).value_counts())

source1_entity_id
True    2206821
Name: count, dtype: int64


In [25]:
sample_ids = gt["matched_entity_ids"].dropna().head(100)

sample_ids

0      S2-681193310,S2-743505751,S3-775321672,S3-1129...
1      S2-249013014,S2-197070651,S3-478195123,S3-3843...
2                 S2-790675320,S2-479876582,S3-878454467
3                  S2-153058913,S2-24659151,S3-679606215
4      S2-478959098,S2-553508714,S2-625774905,S3-7280...
                             ...                        
101    S2-718026833,S2-752280842,S3-255051180,S3-6903...
102    S2-955993024,S2-339477476,S2-981272489,S3-9637...
103    S2-26500789,S2-912528411,S3-741223656,S3-64382...
104                                         S2-198723291
105               S2-138434584,S2-180445273,S2-598796961
Name: matched_entity_ids, Length: 100, dtype: object

In [27]:
s2 = gt["matched_entity_ids"].str.contains("S2-", na=False)
s3 = gt["matched_entity_ids"].str.contains("S3-", na=False)

print("S2 only:", (s2 & ~s3).sum())
print("S3 only:", (s3 & ~s2).sum())
print("Both:", (s2 & s3).sum())
print("No match:", (~s2 & ~s3).sum())

S2 only: 143029
S3 only: 164498
Both: 1776047
No match: 123247


In [28]:
s2_count = gt["matched_entity_ids"].str.count("S2-").fillna(0)
s3_count = gt["matched_entity_ids"].str.count("S3-").fillna(0)

print("S2 matches per S1:")
print(s2_count.describe())

print("\nS3 matches per S1:")
print(s3_count.describe())

S2 matches per S1:
count    2.206821e+06
mean     1.673728e+00
std      1.112813e+00
min      0.000000e+00
25%      1.000000e+00
50%      2.000000e+00
75%      2.000000e+00
max      5.000000e+00
Name: matched_entity_ids, dtype: float64

S3 matches per S1:
count    2.206821e+06
mean     1.787524e+00
std      1.165604e+00
min      0.000000e+00
25%      1.000000e+00
50%      2.000000e+00
75%      3.000000e+00
max      6.000000e+00
Name: matched_entity_ids, dtype: float64


In [29]:
print("\nS2 count distribution:")
print(s2_count.value_counts().sort_index())

print("\nS3 count distribution:")
print(s3_count.value_counts().sort_index())


S2 count distribution:
matched_entity_ids
0.0    287745
1.0    789108
2.0    652779
3.0    333957
4.0    119078
5.0     24154
Name: count, dtype: int64

S3 count distribution:
matched_entity_ids
0.0    266276
1.0    716417
2.0    668375
3.0    372443
4.0    145116
5.0     35378
6.0      2816
Name: count, dtype: int64


# Phase 1 — Exploratory Data Analysis (EDA)

## 1. Dataset Overview

The training dataset contains three source datasets and one ground-truth file.

| Dataset | Rows | Columns |
|---|---:|---:|
| Source 1 | 2,206,821 | 4 |
| Source 2 | 5,034,616 | 4 |
| Source 3 | 5,285,603 | 4 |
| Ground Truth | 2,206,821 | 2 |

### Source columns

- `entity_id`
- `business_name`
- `business_address`
- `country`

### Ground Truth columns

- `source1_entity_id`
- `matched_entity_ids`

`matched_entity_ids` contains comma-separated IDs from Source 2 and/or Source 3 that represent the same real-world entity as the Source-1 entity.

---

## 2. Missing Value Analysis

| Dataset | Missing Business Name | Missing Address |
|---|---:|---:|
| Source 1 | 0 | 0 |
| Source 2 | 2 (≈0%) | 168,967 (3.36%) |
| Source 3 | 13 (≈0%) | 175,916 (3.33%) |

Ground Truth contains **123,247 empty match entries (5.58%)**.

These represent Source-1 entities with **no matching entity** in Source 2 or Source 3, rather than ordinary missing data.

### Finding

Source 1 is complete, while approximately 3.3% of Source 2 and Source 3 addresses are unavailable. Therefore, address information cannot always be relied upon during entity matching.

---

## 3. Duplicate Analysis

### Entity-level duplicates

- Source 1: 0 duplicate rows, 0 duplicate entity IDs
- Source 2: 0 duplicate rows, 0 duplicate entity IDs
- Source 3: 0 duplicate rows, 0 duplicate entity IDs
- Ground Truth: 0 duplicate rows

However, business names are frequently repeated.

| Source | Duplicate Business Names |
|---|---:|
| Source 1 | 667,592 |
| Source 2 | 632,607 |
| Source 3 | 633,994 |

### Finding

`business_name` is **not a unique identifier**.

Examples of highly repeated names include:

- `Primary Care Group`
- `Ear Nose & Throat Group`
- `Physical Therapy`
- `Urgent Care`

Therefore, matching using business name alone can produce many false candidates.

---

## 4. Business Name Distribution

| Source | Unique Names | Names Appearing More Than Once |
|---|---:|---:|
| Source 1 | 1,539,229 | 177,793 |
| Source 2 | 4,402,008 | 239,778 |
| Source 3 | 4,651,608 | 258,275 |

Average business-name length:

| Source | Mean Length |
|---|---:|
| Source 1 | 24.0 |
| Source 2 | 25.1 |
| Source 3 | 25.2 |

### Finding

Business-name lengths are similar across all three sources, but repeated/generic names are common. Name similarity will therefore need to be combined with other attributes.

---

## 5. Country Distribution

| Source | US | India |
|---|---:|---:|
| Source 1 | 1,323,633 | 883,188 |
| Source 2 | 3,016,817 | 2,017,799 |
| Source 3 | 3,170,056 | 2,115,547 |

All three sources have approximately the same distribution:

- ~60% US
- ~40% India

### Finding

`country` is consistent across the three sources and can potentially be used as a strong blocking/filtering attribute during candidate generation.

---

## 6. Address Analysis

Average address length:

| Source | Mean | Median | Minimum | Maximum |
|---|---:|---:|---:|---:|
| Source 1 | 52.1 | 41 | 11 | 256 |
| Source 2 | 46.2 | 37 | 0 | 249 |
| Source 3 | 46.7 | 42 | 0 | 240 |

### Finding

Source 1 has no missing addresses, while Source 2 and Source 3 contain missing addresses.

Address information is therefore useful for matching, but cannot be treated as universally available.

---

## 7. Ground Truth — Number of Matches per Source-1 Entity

Each Source-1 entity has one Ground Truth row.

Number of matches ranges from **0 to 11**.

| Number of Matches | Source-1 Entities |
|---:|---:|
| 0 | 123,247 |
| 1 | 119,157 |
| 2 | 375,212 |
| 3 | 530,841 |
| 4 | 484,115 |
| 5 | 321,957 |
| 6 | 164,868 |
| 7 | 63,968 |
| 8 | 18,680 |
| 9 | 4,205 |
| 10 | 534 |
| 11 | 37 |

### Finding

This is a **one-to-many entity matching problem**, not a simple one-to-one matching problem.

- 123,247 (~5.58%) have no match.
- The most common case is 3 matches.
- Most entities have 2–5 matches.
- Maximum observed matches = 11.

---

## 8. Source-2 vs Source-3 Match Distribution

| Match Type | Source-1 Entities |
|---|---:|
| S2 only | 143,029 |
| S3 only | 164,498 |
| Both S2 and S3 | 1,776,047 |
| No match | 123,247 |

### Finding

Approximately **80.5%** of Source-1 entities have matches in **both Source 2 and Source 3**.

This indicates that information from both sources can potentially provide complementary evidence for entity matching.

---

## 9. Matches per Source

### Source 2

| Matches per S1 | Entities |
|---:|---:|
| 0 | 287,745 |
| 1 | 789,108 |
| 2 | 652,779 |
| 3 | 333,957 |
| 4 | 119,078 |
| 5 | 24,154 |

Mean = **1.67 matches/S1**  
Median = **2**  
Maximum = **5**

### Source 3

| Matches per S1 | Entities |
|---:|---:|
| 0 | 266,276 |
| 1 | 716,417 |
| 2 | 668,375 |
| 3 | 372,443 |
| 4 | 145,116 |
| 5 | 35,378 |
| 6 | 2,816 |

Mean = **1.79 matches/S1**  
Median = **2**  
Maximum = **6**

### Finding

Source 3 contributes slightly more matches per Source-1 entity on average than Source 2.

Typical contribution:

- Source 2 → 1–2 matches
- Source 3 → 1–3 matches

---

## 10. Ground Truth Integrity Check

All Ground Truth `source1_entity_id` values were verified against Source 1.

```text
2,206,821 / 2,206,821 = 100%

In [30]:
sample_gt = gt.head(10000)

sample_gt.head()

,source1_entity_id,matched_entity_ids,s2_matches,s3_matches
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129...",2,3
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843...",2,2
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467",2,1
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215",2,1
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280...",3,3


In [32]:
row = gt.iloc[0]

print("S1 ID:", row["source1_entity_id"])
print("Matches:", row["matched_entity_ids"])

S1 ID: S1-965667
Matches: S2-681193310,S2-743505751,S3-775321672,S3-11291185,S3-860443364


In [33]:
s1[s1["entity_id"] == "S1-965667"]

,entity_id,business_name,business_address,country
1286323,S1-965667,Maure Williams Colombier Inc,"85 Wayne Avenue, Ticonderoga, NY",US


In [37]:
ids = [
    "S2-681193310", "S2-743505751",
    "S3-775321672", "S3-11291185", "S3-860443364"
]

pd.concat([
    source2[source2["entity_id"].isin(ids)],
    source3[source3["entity_id"].isin(ids)]
])

,entity_id,business_name,business_address,country
1376739,S2-681193310,Maure Wilblims Colombier Inc,NaN,US
2003754,S2-743505751,Maure Williams Colombier,NaN,US
63830,S3-860443364,Maure Williams Inc Center,NaN,US
1217885,S3-775321672,Dréxkor,"85 Wanye Avenue, Ticonderoga Townshiip, New York",US
5223525,S3-11291185,maurewilliamscolombier.com,"Wayne Ave, Ticonderoga Townshiip, New York",US


#Major EDA finding

Exact string matching will not be sufficient.

In [39]:
source1 = s1
source2 = source2
source3 = source3

In [40]:
source1[source1["entity_id"] == "S1-965667"]

,entity_id,business_name,business_address,country
1286323,S1-965667,Maure Williams Colombier Inc,"85 Wayne Avenue, Ticonderoga, NY",US


# now  below we are doing random choices 

In [42]:
import random

sample = gt.sample(10, random_state=42)

for _, row in sample.iterrows():
    print("\n" + "="*80)
    print("S1:", row["source1_entity_id"])
    print("Matches:", row["matched_entity_ids"])

    display(
        source1[source1["entity_id"] == row["source1_entity_id"]]
    )

    match_string = row["matched_entity_ids"]
    ids = match_string.split(",") if pd.notna(match_string) and match_string != "" else []

    display(pd.concat([
        source2[source2["entity_id"].isin(ids)],
        source3[source3["entity_id"].isin(ids)]
    ]))


S1: S1-384734976
Matches: S2-685146894,S2-908470079,S3-109845102,S3-176459320


,entity_id,business_name,business_address,country
1645812,S1-384734976,"FQ Social, Inc","1212 Old Rte 34, Sandwich, IL",US


,entity_id,business_name,business_address,country
564407,S2-685146894,"FQ Social, Inc.","1212 OLD RTE 34, SANDWICH, IL",US
585013,S2-908470079,"FQ Social, #42836","1212 OLD RTE 34, SANDWICH, IL",US
1795329,S3-109845102,"FQ Social, Ínc","1212 Old Rte 34, Sandwich, Illinois",US
4635538,S3-176459320,"Arcxylo DBA FQ Social, Inc","1212 Old Rte 34, Sandwich, IL",US



S1: S1-574038342
Matches: S2-133116927,S2-180265553,S2-201976425


,entity_id,business_name,business_address,country
612837,S1-574038342,Innovative Consultants,"15 16 Indrapuri Madhubancolony Karvenagar, Pun...",India


...

,entity_id,business_name,business_address,country
520132,S1-281641466,Life Construction Private Limited,"House No- 24/153, 3Rd Floor Blk-24, Delhi, Nor...",India


,entity_id,business_name,business_address,country
1388626,S2-559704787,लाइफ कंस्ट्रक्शन प्राइवेट लिमिटेड,"HOUSE NO- 24/153, DELHI, NORTH DELHI, Delhi",India
1757245,S3-772912089,Shri Life Construction Private,"House No- 24/153, 3Rd Floor Blk-24, Delhi, Nor...",India



S1: S1-608038725
Matches: S2-895472957,S2-192681436,S2-217803628,S3-993567829


,entity_id,business_name,business_address,country
473783,S1-608038725,Anchor Staking Inc,"2330 Central Avenue, Kansas City, KS",US


,entity_id,business_name,business_address,country
358703,S2-192681436,anchor staking inc,"2330 CETRAL AVENUE, KANSAS CITY, KS",US
1660665,S2-895472957,Anchor Stáking Inc,"CENTRAL AVE, KANSAS CITY, KS",US
1951598,S2-217803628,ANCHOR STAKIFNG INC,"2330 CENTRAL AVE, KANSAS CITY, KS",US
761061,S3-993567829,Anchorstaking.Com,"#2330 Central Ave, Kansas City, Kansas",US


# Pattern found:

### Name variations — typos, case, punctuation, suffixes.

### Address variations — abbreviations, ordering, spelling errors.

### Missing addresses — especially S2/S3.
### Domain names can represent businesses.
### Multilingual names occur.
### Country is consistent and useful as a constraint.
### One S1 → multiple S2/S3 matches is common.
### Some true matches have weak name similarity but strong address similarity.

In [11]:
from rapidfuzz.fuzz import ratio
from tqdm import tqdm

s1_lookup = source1.set_index("entity_id")["business_name"].to_dict()
s2_lookup = source2.set_index("entity_id")["business_name"].to_dict()
s3_lookup = source3.set_index("entity_id")["business_name"].to_dict()

sample = gt[
    gt["matched_entity_ids"].notna() &
    (gt["matched_entity_ids"].str.strip() != "")
].sample(
    10000,
    random_state=42
)

results = []

for _, row in tqdm(
    sample.iterrows(),
    total=len(sample),
    desc="Calculating name similarities"
):

    s1_id = row["source1_entity_id"]

    s1_name = s1_lookup.get(s1_id)

    if s1_name is None:
        continue

    for match_id in str(row["matched_entity_ids"]).split(","):

        match_id = match_id.strip()

        if match_id.startswith("S2-"):
            name = s2_lookup.get(match_id)
        else:
            name = s3_lookup.get(match_id)

        if name is not None:
            results.append(
                ratio(str(s1_name).lower(), str(name).lower())
            )



print("\nFinished!")
print("Number of matched pairs:", len(results))
print("Mean name similarity:", sum(results) / len(results))
print("Minimum:", min(results))
print("Maximum:", max(results))

Calculating name similarities: 100%|███████████████████████████████████████████| 10000/10000 [00:02<00:00, 4108.28it/s]


Finished!
Number of matched pairs: 36703
Mean name similarity: 77.96565971561394
Minimum: 3.8461538461538436
Maximum: 100.0


In [52]:


print("Name Similarity Distribution")
print("=" * 40)

for p in [1, 5, 10, 25, 50, 75, 90, 95, 99]:
    print(f"P{p:02d}: {np.percentile(results, p):.2f}")

print(f"\nMean   : {np.mean(results):.2f}")
print(f"Median : {np.median(results):.2f}")
print(f"Min    : {np.min(results):.2f}")
print(f"Max    : {np.max(results):.2f}")

Name Similarity Distribution
P01: 8.22
P05: 10.53
P10: 44.44
P25: 71.43
P50: 86.21
P75: 94.74
P90: 100.00
P95: 100.00
P99: 100.00

Mean   : 77.97
Median : 86.21
Min    : 3.85
Max    : 100.00


In [53]:
thresholds = [50, 60, 70, 75, 80, 85, 90, 95, 100]

print("Threshold → % of true matches retained")
print("=" * 45)

for t in thresholds:
    retained = sum(x >= t for x in results)
    percentage = retained / len(results) * 100

    print(f">= {t:3d}: {percentage:6.2f}%")

Threshold → % of true matches retained
>=  50:  89.29%
>=  60:  85.34%
>=  70:  76.74%
>=  75:  70.49%
>=  80:  63.20%
>=  85:  52.84%
>=  90:  40.52%
>=  95:  24.28%
>= 100:  10.48%


In [54]:
thresholds = [50, 60, 70, 80, 85, 90, 95, 99, 100]

print("True-match retention by threshold")
print("=" * 45)

for t in thresholds:
    retained = sum(x >= t for x in results)
    percentage = retained / len(results) * 100

    print(
        f">= {t:3d}: "
        f"{retained:6d} / {len(results)} "
        f"({percentage:6.2f}%)"
    )

True-match retention by threshold
>=  50:  32771 / 36703 ( 89.29%)
>=  60:  31323 / 36703 ( 85.34%)
>=  70:  28165 / 36703 ( 76.74%)
>=  80:  23198 / 36703 ( 63.20%)
>=  85:  19393 / 36703 ( 52.84%)
>=  90:  14872 / 36703 ( 40.52%)
>=  95:   8912 / 36703 ( 24.28%)
>=  99:   3846 / 36703 ( 10.48%)
>= 100:   3845 / 36703 ( 10.48%)


# Negative pair analysis

In [57]:
from rapidfuzz.fuzz import ratio
from tqdm import tqdm
import random
import pandas as pd

random.seed(42)

# Take only 20,000 random target rows
s2_sample = source2.sample(20000, random_state=42)
s3_sample = source3.sample(20000, random_state=43)

# Combine targets
targets = pd.concat([
    s2_sample[["entity_id", "business_name", "country"]],
    s3_sample[["entity_id", "business_name", "country"]]
], ignore_index=True)

# Ground-truth lookup only for sampled S1 entities
gt_lookup = gt.set_index("source1_entity_id")["matched_entity_ids"].to_dict()

negative_results = []

# Sample 10k S1 entities
s1_sample = source1.sample(10000, random_state=42)

for _, row in tqdm(
    s1_sample.iterrows(),
    total=len(s1_sample),
    desc="Calculating negative similarities"
):

    # Same-country candidates
    candidates = targets[targets["country"] == row["country"]]

    if len(candidates) == 0:
        continue

    candidate = candidates.sample(1).iloc[0]

    # Make sure it's not a known match
    true_ids = str(gt_lookup.get(row["entity_id"], ""))

    if candidate["entity_id"] in true_ids.split(","):
        continue

    negative_results.append(
        ratio(
            str(row["business_name"]).lower(),
            str(candidate["business_name"]).lower()
        )
    )

print("\nFinished!")
print("Negative pairs:", len(negative_results))
print(
    "Mean negative similarity:",
    sum(negative_results) / len(negative_results)
)

Calculating negative similarities: 100%|██████████████████████████████████████████████████████| 10000/10000 [02:00<00:00, 83.11it/s]


Finished!
Negative pairs: 10000
Mean negative similarity: 32.43143823204454


In [8]:
import numpy as np
import pandas as pd

# ============================================================
# CONFIG
# ============================================================

N_NEGATIVES = 10_000
RANDOM_SEED = 42

rng = np.random.default_rng(RANDOM_SEED)

print("Starting memory-efficient negative pair generation...")


# ============================================================
# 1. GET COUNTRY INDICES
#    IMPORTANT: We DO NOT create filtered DataFrames.
# ============================================================

s2_us_idx = source2.index[
    source2["country"].eq("US")
].to_numpy()

s2_india_idx = source2.index[
    source2["country"].eq("India")
].to_numpy()

s3_us_idx = source3.index[
    source3["country"].eq("US")
].to_numpy()

s3_india_idx = source3.index[
    source3["country"].eq("India")
].to_numpy()


print("\nCountry index sizes:")
print("S2 US:", len(s2_us_idx))
print("S2 India:", len(s2_india_idx))
print("S3 US:", len(s3_us_idx))
print("S3 India:", len(s3_india_idx))


# ============================================================
# 2. DETERMINE HOW MANY NEGATIVES TO GENERATE
# ============================================================

n = min(
    N_NEGATIVES,
    len(source2),
    len(source3)
)

print(f"\nGenerating {n:,} negative pairs...")


# ============================================================
# 3. SAMPLE ROW INDICES
# ============================================================

# We generate approximately half US and half India
n_us = n // 2
n_india = n - n_us


# ------------------------------------------------------------
# US
# ------------------------------------------------------------

us_s2 = rng.choice(
    s2_us_idx,
    size=min(n_us, len(s2_us_idx)),
    replace=False
)

us_s3 = rng.choice(
    s3_us_idx,
    size=min(n_us, len(s3_us_idx)),
    replace=False
)


# ------------------------------------------------------------
# India
# ------------------------------------------------------------

india_s2 = rng.choice(
    s2_india_idx,
    size=min(n_india, len(s2_india_idx)),
    replace=False
)

india_s3 = rng.choice(
    s3_india_idx,
    size=min(n_india, len(s3_india_idx)),
    replace=False
)


# ============================================================
# 4. COMBINE INDICES
# ============================================================

s2_indices = np.concatenate([
    us_s2,
    india_s2
])

s3_indices = np.concatenate([
    us_s3,
    india_s3
])


# ============================================================
# 5. SHUFFLE PAIRINGS
#    This ensures S2 and S3 rows are not simply aligned.
# ============================================================

rng.shuffle(s3_indices)


# ============================================================
# 6. BUILD NEGATIVE PAIRS
#    ONLY THE SMALL RESULT IS COPIED.
# ============================================================

negative_pairs = pd.DataFrame({
    "s2_idx": s2_indices,
    "s3_idx": s3_indices
})


# ============================================================
# 7. VERIFY COUNTRY MATCHING
# ============================================================

negative_pairs["s2_country"] = source2.loc[
    negative_pairs["s2_idx"],
    "country"
].to_numpy()

negative_pairs["s3_country"] = source3.loc[
    negative_pairs["s3_idx"],
    "country"
].to_numpy()


negative_pairs["same_country"] = (
    negative_pairs["s2_country"]
    == negative_pairs["s3_country"]
)


# ============================================================
# 8. RESULTS
# ============================================================

print("\n" + "=" * 50)
print("NEGATIVE PAIR GENERATION COMPLETE")
print("=" * 50)

print("Total negative pairs:", len(negative_pairs))

print(
    "Same-country pairs:",
    negative_pairs["same_country"].sum()
)

print(
    "Different-country pairs:",
    (~negative_pairs["same_country"]).sum()
)

print("\nCountry distribution:")
print(
    negative_pairs["s2_country"].value_counts()
)

print("\nSample:")
print(
    negative_pairs.head()
)

Starting memory-efficient negative pair generation...

Country index sizes:
S2 US: 3016817
S2 India: 2017799
S3 US: 3170056
S3 India: 2115547

Generating 10,000 negative pairs...

NEGATIVE PAIR GENERATION COMPLETE
Total negative pairs: 10000
Same-country pairs: 5098
Different-country pairs: 4902

Country distribution:
s2_country
US       5000
India    5000
Name: count, dtype: int64

Sample:
    s2_idx   s3_idx s2_country s3_country  same_country
0  1255452  4713875         US         US          True
1  4123207  4153963         US      India         False
2  4968485  2350049         US         US          True
3   895154  1416733         US      India         False
4   659426  3604030         US         US          True


In [6]:
import numpy as np
from rapidfuzz.fuzz import ratio
from tqdm import tqdm

N_NEGATIVES = 10_000
rng = np.random.default_rng(42)

# Country indices — only integer indices
s2_us = source2.index[source2["country"].eq("US")].to_numpy()
s2_india = source2.index[source2["country"].eq("India")].to_numpy()

s3_us = source3.index[source3["country"].eq("US")].to_numpy()
s3_india = source3.index[source3["country"].eq("India")].to_numpy()

# Sample S1
s1_indices = rng.choice(
    len(source1),
    size=N_NEGATIVES,
    replace=False
)

negative_results = []

for idx in tqdm(
    s1_indices,
    desc="Calculating negative similarities"
):

    country = source1.iloc[idx]["country"]

    # Pick S2 or S3, but SAME COUNTRY
    if rng.random() < 0.5:

        candidates = (
            s2_us if country == "US"
            else s2_india
        )

        target_idx = rng.choice(candidates)
        target_name = source2.iloc[target_idx]["business_name"]

    else:

        candidates = (
            s3_us if country == "US"
            else s3_india
        )

        target_idx = rng.choice(candidates)
        target_name = source3.iloc[target_idx]["business_name"]

    s1_name = source1.iloc[idx]["business_name"]

    negative_results.append(
        ratio(
            str(s1_name).lower(),
            str(target_name).lower()
        )
    )

print("\nFinished!")
print("Negative pairs:", len(negative_results))
print(
    "Mean negative similarity:",
    np.mean(negative_results)
)

Calculating negative similarities: 100%|███████████████████████████████████████| 10000/10000 [00:05<00:00, 1729.30it/s]


Finished!
Negative pairs: 10000
Mean negative similarity: 32.46133516016396


In [12]:
import numpy as np

percentiles = [1, 5, 10, 25, 50, 75, 90, 95, 99]

print(f"{'Percentile':<12}{'True Match':<15}{'Negative':<15}")
print("-" * 42)

for p in percentiles:
    true_score = np.percentile(results, p)
    neg_score = np.percentile(negative_results, p)

    print(f"P{p:<11}{true_score:<15.2f}{neg_score:<15.2f}")

Percentile  True Match     Negative       
------------------------------------------
P1          8.22           5.00           
P5          10.53          9.52           
P10         44.44          17.64          
P25         71.43          25.93          
P50         86.21          32.00          
P75         94.74          38.46          
P90         100.00         47.46          
P95         100.00         54.55          
P99         100.00         66.67          


# Phase 1 — Exploratory Data Analysis (EDA)

## Dataset Structure

| Dataset | Rows | Columns |
|---|---:|---:|
| Source 1 | 2,206,821 | 4 |
| Source 2 | 5,034,616 | 4 |
| Source 3 | 5,285,603 | 4 |
| Ground Truth | 2,206,821 | 2 |

## Key Findings

### Missing Values
- S1 has no missing business names or addresses.
- S2: 2 missing names, 168,967 missing addresses (3.36%).
- S3: 13 missing names, 175,916 missing addresses (3.33%).
- Ground Truth has 123,247 entities (5.58%) with no known match.

### Duplicates
- No duplicate rows or entity IDs in S1, S2, or S3.
- Business names are heavily duplicated.
- Therefore, `business_name` is not a unique identifier.

### Country
- All sources are approximately 60% US and 40% India.
- `country` is a useful potential blocking signal.

### Address
- S1 addresses are always present.
- S2/S3 have ~3.3% missing addresses.
- Address is useful but cannot always be relied upon.

### Ground Truth
- One-to-many matching problem.
- Each S1 entity has 0–11 valid matches.
- 123,247 S1 entities have no known match.
- 100% of Ground Truth S1 IDs exist in S1.

### Manual True-Match Analysis
Observed variations include:
- spelling errors
- capitalization/punctuation differences
- abbreviations
- company suffix changes
- word additions/removals
- address formatting differences
- partial/missing addresses
- domain names
- multilingual names
- substantially different names

Therefore, exact string matching is insufficient.

---

# Name Similarity Analysis

## True Matches

Sample:
- 10,000 S1 entities
- 36,703 true matched pairs

Using `rapidfuzz.fuzz.ratio`:

| Metric | Score |
|---|---:|
| Mean | 77.97 |
| Median | 86.21 |
| Minimum | 3.85 |
| Maximum | 100.00 |
| P10 | 44.44 |
| P25 | 71.43 |
| P75 | 94.74 |
| P90 | 100.00 |
| P95 | 100.00 |
| P99 | 100.00 |

### True-match retention

| Threshold | Retained |
|---:|---:|
| >= 50 | 89.29% |
| >= 60 | 85.34% |
| >= 70 | 76.74% |
| >= 80 | 63.20% |
| >= 85 | 52.84% |
| >= 90 | 40.52% |
| >= 95 | 24.28% |
| >= 99 | 10.48% |
| = 100 | 10.48% |

## Negative Pairs

10,000 random **same-country** negative pairs were generated from the full S2/S3 populations.

Mean negative similarity:

**32.46**

| Percentile | True Match | Negative |
|---|---:|---:|
| P1 | 8.22 | 5.00 |
| P5 | 10.53 | 9.52 |
| P10 | 44.44 | 17.64 |
| P25 | 71.43 | 25.93 |
| P50 | 86.21 | 32.00 |
| P75 | 94.74 | 38.46 |
| P90 | 100.00 | 47.46 |
| P95 | 100.00 | 54.55 |
| P99 | 100.00 | 66.67 |

## Current Conclusion

Business-name similarity is a strong matching signal, but it is not sufficient by itself.

The eventual matcher should combine:

**Business Name + Address + Country + additional signals**

In [13]:
from rapidfuzz.fuzz import ratio
from tqdm import tqdm
import numpy as np

# Fast lookups for only required columns
s1_address = source1.set_index("entity_id")["business_address"].to_dict()
s2_address = source2.set_index("entity_id")["business_address"].to_dict()
s3_address = source3.set_index("entity_id")["business_address"].to_dict()

address_results = []

for _, row in tqdm(
    sample.iterrows(),
    total=len(sample),
    desc="Calculating address similarities"
):
    s1_id = row["source1_entity_id"]

    s1_addr = s1_address.get(s1_id)

    if pd.isna(s1_addr):
        continue

    for match_id in str(row["matched_entity_ids"]).split(","):

        match_id = match_id.strip()

        if match_id.startswith("S2-"):
            addr = s2_address.get(match_id)
        else:
            addr = s3_address.get(match_id)

        # Skip missing addresses
        if pd.isna(addr):
            continue

        address_results.append(
            ratio(
                str(s1_addr).lower(),
                str(addr).lower()
            )
        )

print("\nFinished!")
print("Matched address pairs:", len(address_results))
print("Mean:", np.mean(address_results))
print("Median:", np.median(address_results))
print("Min:", np.min(address_results))
print("Max:", np.max(address_results))

Calculating address similarities: 100%|████████████████████████████████████████| 10000/10000 [00:02<00:00, 4546.14it/s]


Finished!
Matched address pairs: 35088
Mean: 78.77712802354696
Median: 83.56164383561644
Min: 13.913043478260867
Max: 100.0


In [14]:
percentiles = [1, 5, 10, 25, 50, 75, 90, 95, 99]

print(f"{'Percentile':<12}{'Address Similarity':<20}")
print("-" * 32)

for p in percentiles:
    score = np.percentile(address_results, p)
    print(f"P{p:<11}{score:.2f}")

Percentile  Address Similarity  
--------------------------------
P1          32.96
P5          45.00
P10         51.16
P25         68.49
P50         83.56
P75         92.17
P90         97.83
P95         100.00
P99         100.00


In [16]:
from rapidfuzz.fuzz import ratio
from tqdm import tqdm
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# 10k S1 sample
s1_sample = source1.sample(10_000, random_state=42)

# Country indices
s2_us = source2.index[source2["country"].eq("US")].to_numpy()
s2_india = source2.index[source2["country"].eq("India")].to_numpy()
s3_us = source3.index[source3["country"].eq("US")].to_numpy()
s3_india = source3.index[source3["country"].eq("India")].to_numpy()

negative_address_results = []

for idx in tqdm(
    s1_sample.index,
    total=len(s1_sample),
    desc="Calculating negative address similarities"
):
    s1_row = source1.loc[idx]
    country = s1_row["country"]

    if country == "US":
        candidates = s2_us if rng.random() < 0.5 else s3_us
    else:
        candidates = s2_india if rng.random() < 0.5 else s3_india

    target_idx = rng.choice(candidates)

    target_df = source2 if candidates is s2_us or candidates is s2_india else source3
    target = target_df.loc[target_idx]

    if pd.isna(s1_row["business_address"]) or pd.isna(target["business_address"]):
        continue

    negative_address_results.append(
        ratio(
            str(s1_row["business_address"]).lower(),
            str(target["business_address"]).lower()
        )
    )

print("\nFinished!")
print("Negative address pairs:", len(negative_address_results))
print("Mean:", np.mean(negative_address_results))

Calculating negative address similarities: 100%|███████████████████████████████| 10000/10000 [00:05<00:00, 1997.32it/s]


Finished!
Negative address pairs: 9658
Mean: 35.687699646680755


In [17]:
percentiles = [1, 5, 10, 25, 50, 75, 90, 95, 99]

print(f"{'Percentile':<12}{'True Address':<15}{'Negative Address':<18}")
print("-" * 45)

for p in percentiles:
    true_score = np.percentile(address_results, p)
    neg_score = np.percentile(negative_address_results, p)

    print(f"P{p:<11}{true_score:<15.2f}{neg_score:<18.2f}")

Percentile  True Address   Negative Address  
---------------------------------------------
P1          32.96          21.43             
P5          45.00          26.23             
P10         51.16          28.57             
P25         68.49          32.00             
P50         83.56          35.48             
P75         92.17          39.08             
P90         97.83          43.04             
P95         100.00         45.90             
P99         100.00         52.63             


In [19]:
from rapidfuzz.fuzz import ratio
from tqdm import tqdm
import numpy as np
import pandas as pd

# Lookups
s1_names = source1.set_index("entity_id")["business_name"].to_dict()
s1_addresses = source1.set_index("entity_id")["business_address"].to_dict()

s2_names = source2.set_index("entity_id")["business_name"].to_dict()
s2_addresses = source2.set_index("entity_id")["business_address"].to_dict()

s3_names = source3.set_index("entity_id")["business_name"].to_dict()
s3_addresses = source3.set_index("entity_id")["business_address"].to_dict()

combined_results = []

for _, row in tqdm(
    sample.iterrows(),
    total=len(sample),
    desc="Calculating combined similarity"
):

    s1_id = row["source1_entity_id"]

    s1_name = s1_names.get(s1_id)
    s1_addr = s1_addresses.get(s1_id)

    if pd.isna(s1_name) or pd.isna(s1_addr):
        continue

    for match_id in str(row["matched_entity_ids"]).split(","):

        match_id = match_id.strip()

        if match_id.startswith("S2-"):
            target_name = s2_names.get(match_id)
            target_addr = s2_addresses.get(match_id)
        else:
            target_name = s3_names.get(match_id)
            target_addr = s3_addresses.get(match_id)

        # Address may be missing
        if pd.isna(target_name) or pd.isna(target_addr):
            continue

        name_score = ratio(
            str(s1_name).lower(),
            str(target_name).lower()
        )

        address_score = ratio(
            str(s1_addr).lower(),
            str(target_addr).lower()
        )

        combined_score = (
            0.5 * name_score +
            0.5 * address_score
        )

        combined_results.append(combined_score)

print("\nFinished!")
print("Pairs:", len(combined_results))
print("Mean:", np.mean(combined_results))
print("Median:", np.median(combined_results))
print("Min:", np.min(combined_results))
print("Max:", np.max(combined_results))

Calculating combined similarity: 100%|█████████████████████████████████████████| 10000/10000 [00:03<00:00, 2645.59it/s]



Finished!
Pairs: 35088
Mean: 78.18058556030817
Median: 82.05923469108053
Min: 14.102564102564102
Max: 100.0


In [20]:
percentiles = [1, 5, 10, 25, 50, 75, 90, 95, 99]

print(f"{'P':<6}{'Name':<12}{'Address':<12}{'Combined':<12}")
print("-" * 42)

for p in percentiles:
    print(
        f"P{p:<4}"
        f"{np.percentile(results, p):<12.2f}"
        f"{np.percentile(address_results, p):<12.2f}"
        f"{np.percentile(combined_results, p):<12.2f}"
    )

P     Name        Address     Combined    
------------------------------------------
P1   8.22        32.96       28.43       
P5   10.53       45.00       45.89       
P10  44.44       51.16       55.01       
P25  71.43       68.49       71.15       
P50  86.21       83.56       82.06       
P75  94.74       92.17       89.47       
P90  100.00      97.83       94.04       
P95  100.00      100.00      96.15       
P99  100.00      100.00      98.94       


In [21]:
weights = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

print(f"{'Name W':<10}{'Address W':<12}{'Mean':<12}{'Median':<12}")
print("-" * 46)

for w in weights:

    a = 1 - w

    scores = [
        w * n + a * ad
        for n, ad in zip(results[:len(address_results)], address_results)
    ]

    print(
        f"{w:<10.1f}"
        f"{a:<12.1f}"
        f"{np.mean(scores):<12.2f}"
        f"{np.median(scores):<12.2f}"
    )

Name W    Address W   Mean        Median      
----------------------------------------------
0.2       0.8         78.63       82.01       
0.3       0.7         78.56       81.86       
0.4       0.6         78.49       81.79       
0.5       0.5         78.42       81.77       
0.6       0.4         78.35       82.14       
0.7       0.3         78.28       83.03       
0.8       0.2         78.21       84.33       


In [22]:
# Make equal-length arrays
n = min(len(results), len(address_results))
nn = min(len(negative_results), len(negative_address_results))

print(f"{'Name W':<10}{'Addr W':<10}{'True Mean':<12}{'Neg Mean':<12}{'Gap':<10}")
print("-" * 54)

for w in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:

    a = 1 - w

    true_scores = (
        w * np.array(results[:n]) +
        a * np.array(address_results[:n])
    )

    neg_scores = (
        w * np.array(negative_results[:nn]) +
        a * np.array(negative_address_results[:nn])
    )

    true_mean = np.mean(true_scores)
    neg_mean = np.mean(neg_scores)

    print(
        f"{w:<10.1f}"
        f"{a:<10.1f}"
        f"{true_mean:<12.2f}"
        f"{neg_mean:<12.2f}"
        f"{true_mean - neg_mean:<10.2f}"
    )

Name W    Addr W    True Mean   Neg Mean    Gap       
------------------------------------------------------
0.2       0.8       78.63       35.05       43.58     
0.3       0.7       78.56       34.73       43.83     
0.4       0.6       78.49       34.42       44.08     
0.5       0.5       78.42       34.10       44.32     
0.6       0.4       78.35       33.78       44.57     
0.7       0.3       78.28       33.46       44.82     
0.8       0.2       78.21       33.14       45.06     


In [23]:
import numpy as np

n = min(len(results), len(address_results))

name_arr = np.array(results[:n])
addr_arr = np.array(address_results[:n])

print("Correlation:", np.corrcoef(name_arr, addr_arr)[0, 1])

Correlation: -0.0038695043811154623


In [24]:
print("Name strong / Address weak:",
      np.mean((name_arr >= 80) & (addr_arr < 50)) * 100, "%")

print("Name weak / Address strong:",
      np.mean((name_arr < 50) & (addr_arr >= 80)) * 100, "%")

print("Both strong:",
      np.mean((name_arr >= 80) & (addr_arr >= 80)) * 100, "%")

print("Both weak:",
      np.mean((name_arr < 50) & (addr_arr < 50)) * 100, "%")

print("\n--- Rescue analysis ---")

print("Average address score when name is weak:",
      addr_arr[name_arr < 50].mean())

print("Average name score when address is weak:",
      name_arr[addr_arr < 50].mean())

Name strong / Address weak: 5.474806201550388 %
Name weak / Address strong: 6.252849977200182 %
Both strong: 36.89295485636115 %
Both weak: 0.9176926584587323 %

--- Rescue analysis ---
Average address score when name is weak: 78.79962945887549
Average name score when address is weak: 77.9401048866519


In [27]:
# Get country directly from the sampled S1 entities
sample_countries = (
    sample["source1_entity_id"]
    .map(source1.set_index("entity_id")["country"])
)

country_results = []

for country, match_ids in zip(
    sample_countries,
    sample["matched_entity_ids"]
):
    country_results.extend(
        [country] * len(str(match_ids).split(","))
    )

n = min(len(country_results), len(results), len(address_results))

country_arr = np.array(country_results[:n])
name_arr2 = np.array(results[:n])
addr_arr2 = np.array(address_results[:n])

for country in ["US", "India"]:

    mask = country_arr == country

    print(f"\n{country}")
    print("Pairs:", mask.sum())
    print("Name mean:", name_arr2[mask].mean())
    print("Address mean:", addr_arr2[mask].mean())
    print("Name median:", np.median(name_arr2[mask]))
    print("Address median:", np.median(addr_arr2[mask]))


US
Pairs: 20933
Name mean: 84.38346085174112
Address mean: 78.67545857668065
Name median: 88.88888888888889
Address median: 83.33333333333334

India
Pairs: 14155
Name mean: 68.71587871164519
Address mean: 78.92748101056587
Name median: 80.95238095238095
Address median: 83.80952380952381


In [28]:
# Recreate country labels for the negative S1 sample
neg_countries = s1_sample["country"].to_numpy()

# Our negative results correspond to the sampled S1 rows,
# except rows with missing addresses were skipped.
# Re-generate with aligned results.

neg_name = []
neg_addr = []
neg_country = []

rng = np.random.default_rng(42)

for idx in s1_sample.index:

    s1 = source1.loc[idx]
    country = s1["country"]

    if country == "US":
        if rng.random() < 0.5:
            target_df = source2
            candidates = s2_us
        else:
            target_df = source3
            candidates = s3_us
    else:
        if rng.random() < 0.5:
            target_df = source2
            candidates = s2_india
        else:
            target_df = source3
            candidates = s3_india

    target = target_df.loc[rng.choice(candidates)]

    if pd.isna(s1["business_address"]) or pd.isna(target["business_address"]):
        continue

    neg_country.append(country)

    neg_name.append(
        ratio(
            str(s1["business_name"]).lower(),
            str(target["business_name"]).lower()
        )
    )

    neg_addr.append(
        ratio(
            str(s1["business_address"]).lower(),
            str(target["business_address"]).lower()
        )
    )

neg_country = np.array(neg_country)
neg_name = np.array(neg_name)
neg_addr = np.array(neg_addr)

for country in ["US", "India"]:

    mask = neg_country == country

    print(f"\n{country}")
    print("Pairs:", mask.sum())
    print("Name mean:", neg_name[mask].mean())
    print("Address mean:", neg_addr[mask].mean())
    print("Name median:", np.median(neg_name[mask]))
    print("Address median:", np.median(neg_addr[mask]))


US
Pairs: 5780
Name mean: 31.457597997527866
Address mean: 36.33134907419435
Name median: 31.57894736842105
Address median: 35.820895522388064

India
Pairs: 3878
Name mean: 33.68706036505466
Address mean: 34.72836656493021
Name median: 33.89830508474576
Address median: 34.98633879781421


In [30]:
# Build only Series lookups — much lighter than DataFrame filtering
s2_addr = source2.set_index("entity_id")["business_address"]
s3_addr = source3.set_index("entity_id")["business_address"]

total = 0
missing = 0

for _, row in tqdm(
    sample.iterrows(),
    total=len(sample),
    desc="Checking true-match addresses"
):
    for match_id in str(row["matched_entity_ids"]).split(","):
        match_id = match_id.strip()

        if match_id.startswith("S2-"):
            addr = s2_addr.get(match_id)
        else:
            addr = s3_addr.get(match_id)

        total += 1

        if pd.isna(addr):
            missing += 1

print("\nTrue-match address availability")
print("=" * 40)
print("Total pairs :", total)
print("Missing     :", missing)
print("Available   :", total - missing)
print("Missing %   :", round(missing / total * 100, 2))

Checking true-match addresses: 100%|████████████████████████████████████████████| 10000/10000 [00:13<00:00, 769.21it/s]


True-match address availability
Total pairs : 36703
Missing     : 1615
Available   : 35088
Missing %   : 4.4


r"""
# Phase 1 — Exploratory Data Analysis (EDA)

## 1. Dataset Structure

| Dataset | Rows | Columns |
|---|---:|---:|
| Source 1 | 2,206,821 | 4 |
| Source 2 | 5,034,616 | 4 |
| Source 3 | 5,285,603 | 4 |
| Ground Truth | 2,206,821 | 2 |

Source columns:
- `entity_id`
- `business_name`
- `business_address`
- `country`

Ground Truth:
- `source1_entity_id`
- `matched_entity_ids`

`matched_entity_ids` contains comma-separated S2/S3 IDs corresponding to the same real-world entity.

---

# 2. Missing Values

| Dataset | Missing Business Name | Missing Address |
|---|---:|---:|
| S1 | 0 | 0 |
| S2 | 2 | 168,967 (3.36%) |
| S3 | 13 | 175,916 (3.33%) |

Ground Truth:
- Empty `matched_entity_ids`: **123,247 (5.58%)**

### Finding

S1 is complete.

Approximately 3.3% of S2/S3 addresses are missing.

---

# 3. Duplicate Analysis

| Source | Duplicate Rows | Duplicate Entity IDs |
|---|---:|---:|
| S1 | 0 | 0 |
| S2 | 0 | 0 |
| S3 | 0 | 0 |

Duplicate business names:

| Source | Duplicate Names |
|---|---:|
| S1 | 667,592 |
| S2 | 632,607 |
| S3 | 633,994 |

### Finding

`business_name` is **not a unique identifier**.

Generic names occur many times, so exact business-name matching cannot reliably identify entities.

---

# 4. Business Name Distribution

| Source | Unique Names | Names Appearing >1 Time |
|---|---:|---:|
| S1 | 1,539,229 | 177,793 |
| S2 | 4,402,008 | 239,778 |
| S3 | 4,651,608 | 258,275 |

Mean name length:

| Source | Mean Length |
|---|---:|
| S1 | 24.0 |
| S2 | 25.1 |
| S3 | 25.2 |

### Finding

Name lengths are broadly similar, but names contain substantial duplication and noise.

---

# 5. Country Distribution

| Source | US | India |
|---|---:|---:|
| S1 | 1,323,633 | 883,188 |
| S2 | 3,016,817 | 2,017,799 |
| S3 | 3,170,056 | 2,115,547 |

Approximately:
- 60% US
- 40% India

### Finding

Country distribution is highly consistent.

`country` is therefore a useful potential **blocking signal**.

---

# 6. Address Analysis

| Source | Mean Length | Median | Min | Max |
|---|---:|---:|---:|---:|
| S1 | 52.1 | 41 | 11 | 256 |
| S2 | 46.2 | 37 | 0 | 249 |
| S3 | 46.7 | 42 | 0 | 240 |

### Finding

S1 addresses are always present.

S2/S3 contain missing addresses.

Address is useful but cannot always be relied upon.

---

# 7. Ground Truth Analysis

Matches per S1 entity:

| Matches | S1 Entities |
|---:|---:|
| 0 | 123,247 |
| 1 | 119,157 |
| 2 | 375,212 |
| 3 | 530,841 |
| 4 | 484,115 |
| 5 | 321,957 |
| 6 | 164,868 |
| 7 | 63,968 |
| 8 | 18,680 |
| 9 | 4,205 |
| 10 | 534 |
| 11 | 37 |

### Finding

This is a **one-to-many entity matching problem**.

Each S1 entity can have **0–11 valid matches**.

---

# 8. Source 2 vs Source 3

| Match Type | S1 Entities |
|---|---:|
| S2 only | 143,029 |
| S3 only | 164,498 |
| Both S2 + S3 | 1,776,047 |
| No match | 123,247 |

Approximately **80.5%** of S1 entities have matches in both S2 and S3.

### S2

- Mean: 1.67 matches/S1
- Median: 2
- Maximum: 5

### S3

- Mean: 1.79 matches/S1
- Median: 2
- Maximum: 6

### Finding

S3 contributes slightly more matches per S1 entity on average.

---

# 9. Ground Truth Integrity

Every Ground Truth `source1_entity_id` exists in Source 1.

**2,206,821 / 2,206,821 = 100%**

### Finding

Ground Truth references are fully covered by S1.

---

# 10. Manual True-Match Analysis

Observed true-match variations:

- spelling errors
- capitalization differences
- punctuation differences
- abbreviations
- company suffix differences
- word additions/removals
- address abbreviations
- address spelling errors
- address formatting differences
- partial addresses
- domain names used as business names
- multilingual names
- substantially different names

Example:

S1:
`Maure Williams Colombier Inc`

True matches included:
- `Maure Wilblims Colombier Inc`
- `Maure Williams Colombier`
- `Maure Williams Inc Center`
- `Dréxkor`
- `maurewilliamscolombier.com`

### Finding

**Exact string matching is insufficient.**

---

# 11. Business Name Similarity — True Matches

Sample:
- 10,000 S1 entities
- 36,703 true matched pairs

Metric: `rapidfuzz.fuzz.ratio`

| Metric | Score |
|---|---:|
| Mean | 77.97 |
| Median | 86.21 |
| Minimum | 3.85 |
| Maximum | 100.00 |
| P10 | 44.44 |
| P25 | 71.43 |
| P75 | 94.74 |
| P90 | 100.00 |
| P95 | 100.00 |
| P99 | 100.00 |

### True-match retention

| Threshold | Retained |
|---:|---:|
| >= 50 | 89.29% |
| >= 60 | 85.34% |
| >= 70 | 76.74% |
| >= 80 | 63.20% |
| >= 85 | 52.84% |
| >= 90 | 40.52% |
| >= 95 | 24.28% |
| >= 99 | 10.48% |
| = 100 | 10.48% |

### Finding

Name similarity is a strong signal, but a high hard threshold would discard many genuine matches.

---

# 12. Negative Name Similarity

Generated **10,000 random same-country negative pairs** from the full S2/S3 populations.

Mean negative name similarity:

**32.46**

| Percentile | True Match | Negative |
|---|---:|---:|
| P1 | 8.22 | 5.00 |
| P5 | 10.53 | 9.52 |
| P10 | 44.44 | 17.64 |
| P25 | 71.43 | 25.93 |
| P50 | 86.21 | 32.00 |
| P75 | 94.74 | 38.46 |
| P90 | 100.00 | 47.46 |
| P95 | 100.00 | 54.55 |
| P99 | 100.00 | 66.67 |

### Finding

Name similarity clearly separates true matches from random same-country negatives, but overlap exists.

---

# 13. Address Similarity — True Matches

True matched pairs with available addresses:

**35,088**

| Metric | Score |
|---|---:|
| Mean | 78.78 |
| Median | 83.56 |
| Minimum | 13.91 |
| P1 | 32.96 |
| P5 | 45.00 |
| P10 | 51.16 |
| P25 | 68.49 |
| P50 | 83.56 |
| P75 | 92.17 |
| P90 | 97.83 |
| P95 | 100.00 |
| P99 | 100.00 |

### Finding

Address similarity is also a strong matching signal.

---

# 14. Negative Address Similarity

Generated same-country negative pairs with available addresses:

**9,658**

Mean negative address similarity:

**35.69**

| Percentile | True | Negative |
|---|---:|---:|
| P1 | 32.96 | 21.43 |
| P5 | 45.00 | 26.23 |
| P10 | 51.16 | 28.57 |
| P25 | 68.49 | 32.00 |
| P50 | 83.56 | 35.48 |
| P75 | 92.17 | 39.08 |
| P90 | 97.83 | 43.04 |
| P95 | 100.00 | 45.90 |
| P99 | 100.00 | 52.63 |

### Finding

Address similarity clearly separates true matches from random same-country negatives.

---

# 15. Name + Address Relationship

Correlation between true-match name and address similarity:

**r = -0.0039**

This is effectively zero.

### Finding

Name and address similarity are largely independent signals.

Therefore, strong name similarity does not necessarily imply strong address similarity, and vice versa.

---

# 16. Signal Complementarity

Among true matches:

| Pattern | Percentage |
|---|---:|
| Name >=80, Address <50 | 5.47% |
| Name <50, Address >=80 | 6.25% |
| Both >=80 | 36.89% |
| Both <50 | 0.92% |

When name similarity was weak (<50):

**Average address similarity = 78.80**

When address similarity was weak (<50):

**Average name similarity = 77.94**

### Finding

Name and address can **rescue each other**.

Only about 0.9% of true matches had both signals below 50.

This supports a combined scoring approach rather than requiring both fields to independently exceed a hard threshold.

---

# 17. Combined Name + Address Score

A simple 50/50 score was tested:

`combined = 0.5 × name + 0.5 × address`

Results:

- Pairs: 35,088
- Mean: 78.18
- Median: 82.06
- Minimum: 14.10
- Maximum: 100

### Weight experiment

| Name Weight | Address Weight | True Mean | Negative Mean | Gap |
|---:|---:|---:|---:|---:|
| 0.2 | 0.8 | 78.63 | 35.05 | 43.58 |
| 0.3 | 0.7 | 78.56 | 34.73 | 43.83 |
| 0.4 | 0.6 | 78.49 | 34.42 | 44.08 |
| 0.5 | 0.5 | 78.42 | 34.10 | 44.32 |
| 0.6 | 0.4 | 78.35 | 33.78 | 44.57 |
| 0.7 | 0.3 | 78.28 | 33.46 | 44.82 |
| 0.8 | 0.2 | 78.21 | 33.14 | 45.06 |

### Finding

Increasing the name weight increased the observed mean separation in this sample.

However, weights should **not yet be finalized** based only on mean gap.

---

# 18. Country-Specific Matching Behavior

True matches:

| Country | Name Mean | Address Mean |
|---|---:|---:|
| US | 84.38 | 78.68 |
| India | 68.72 | 78.93 |

Negative pairs:

| Country | Name Mean | Address Mean |
|---|---:|---:|
| US | 31.46 | 36.33 |
| India | 33.69 | 34.73 |

Mean separation:

| Country | Name Gap | Address Gap |
|---|---:|---:|
| US | 52.93 | 42.34 |
| India | 35.03 | 44.20 |

### Finding

The relative discriminative power differs by country:

- **US:** name is more discriminative.
- **India:** address is more discriminative.

This suggests that country may be useful not only for blocking but potentially for country-specific scoring.

---

# 19. Address Availability in True Matches

Among the 36,703 true matched pairs:

- Address available: **35,088 (95.6%)**
- Address missing: **1,615 (4.4%)**

### Finding

Address is available for approximately **96% of true matches**.

A fallback strategy is therefore required for the ~4.4% of matches where the target address is missing.

---

# Current EDA Conclusions

The evidence so far supports a multi-signal entity-resolution system:

```text
Country
   ↓
Blocking
   ↓
Business Name Similarity
        +
Address Similarity
        +
Additional signals
   ↓
Final Match Score

In [38]:
def address_availability(df, name):
    address = df["business_address"].astype("string")

    usable = address.notna() & address.str.strip().ne("")

    print(f"\n{name}")
    print(f"Rows: {len(df):,}")
    print(f"Usable address: {usable.sum():,} ({usable.mean()*100:.2f}%)")
    print(f"Missing/blank: {(~usable).sum():,} ({(~usable).mean()*100:.2f}%)")

    lengths = address.str.len()
    print(f"Median address length: {lengths.median():.0f}")
    print(f"Mean address length: {lengths.mean():.2f}")


address_availability(source1, "S1")
address_availability(source2, "S2")
address_availability(source3, "S3")


S1
Rows: 2,206,821
Usable address: 2,206,821 (100.00%)
Missing/blank: 0 (0.00%)
Median address length: 41
Mean address length: 52.07

S2
Rows: 5,034,616
Usable address: 4,865,649 (96.64%)
Missing/blank: 168,967 (3.36%)
Median address length: 37
Mean address length: 47.83

S3
Rows: 5,285,603
Usable address: 5,109,687 (96.67%)
Missing/blank: 175,916 (3.33%)
Median address length: 42
Mean address length: 48.32


In [39]:
def country_address_stats(df, source_name):
    temp = df[["country", "business_address"]]

    usable = (
        temp["business_address"]
        .astype("string")
        .str.strip()
        .ne("")
        .fillna(False)
    )

    result = (
        temp.assign(_usable=usable)
        .groupby("country", dropna=False)
        .agg(
            rows=("country", "size"),
            usable_address=("_usable", "sum"),
        )
    )

    result["address_%"] = (
        result["usable_address"] / result["rows"] * 100
    ).round(2)

    result = result.sort_values("rows", ascending=False)

    print(f"\n========== {source_name} ==========")
    display(result)

    return result


s1_country_address = country_address_stats(source1, "S1")
s2_country_address = country_address_stats(source2, "S2")
s3_country_address = country_address_stats(source3, "S3")


========== S1 ==========


,rows,usable_address,address_%
country,,,
US,1323633,1323633,100.0
India,883188,883188,100.0



========== S2 ==========


,rows,usable_address,address_%
country,,,
US,3016817,2905696,96.32
India,2017799,1959953,97.13



========== S3 ==========


,rows,usable_address,address_%
country,,,
US,3170056,3059088,96.5
India,2115547,2050599,96.93


In [40]:
# ============================================================
# PHASE 1 — COUNTRY-SPECIFIC SIGNAL DISTRIBUTIONS
# True matches vs negative pairs
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Inspect the existing pair-level dataframe
# ------------------------------------------------------------

print("Existing variables related to pair analysis:")

for name in [
    "true_pairs",
    "positive_pairs",
    "negative_pairs",
    "pairs",
    "sample_pairs",
    "matched_pairs",
]:
    if name in globals():
        obj = globals()[name]
        print(f"  {name}: {type(obj).__name__}, shape={getattr(obj, 'shape', None)}")

Existing variables related to pair analysis:
  negative_pairs: DataFrame, shape=(10000, 5)


In [41]:
# Show the columns of likely existing pair-level DataFrames

for name in [
    "true_pairs",
    "positive_pairs",
    "negative_pairs",
    "pairs",
    "sample_pairs",
    "matched_pairs",
]:
    if name in globals() and isinstance(globals()[name], pd.DataFrame):
        print(f"\n===== {name} =====")
        print(globals()[name].columns.tolist())
        display(globals()[name].head(3))


===== negative_pairs =====
['s2_idx', 's3_idx', 's2_country', 's3_country', 'same_country']


,s2_idx,s3_idx,s2_country,s3_country,same_country
0,1255452,4713875,US,US,True
1,4123207,4153963,US,India,False
2,4968485,2350049,US,US,True


In [42]:
# ============================================================
# PHASE 1 — CLEAN TRUE-MATCH PAIR DATASET
# Country-specific signal analysis
# ============================================================

from rapidfuzz.fuzz import ratio
import numpy as np
import pandas as pd
from tqdm import tqdm

# ------------------------------------------------------------
# Lightweight Series lookups
# ------------------------------------------------------------

s1_name = source1.set_index("entity_id")["business_name"]
s1_addr = source1.set_index("entity_id")["business_address"]
s1_country = source1.set_index("entity_id")["country"]

s2_name = source2.set_index("entity_id")["business_name"]
s2_addr = source2.set_index("entity_id")["business_address"]

s3_name = source3.set_index("entity_id")["business_name"]
s3_addr = source3.set_index("entity_id")["business_address"]


# ------------------------------------------------------------
# Build aligned TRUE pairs
# ------------------------------------------------------------

true_rows = []

for _, row in tqdm(
    sample.iterrows(),
    total=len(sample),
    desc="Building true-match pairs"
):

    s1_id = row["source1_entity_id"]

    source_name = s1_name.get(s1_id)
    source_addr = s1_addr.get(s1_id)
    country = s1_country.get(s1_id)

    if pd.isna(source_name):
        continue

    for match_id in str(row["matched_entity_ids"]).split(","):

        match_id = match_id.strip()

        if not match_id:
            continue

        if match_id.startswith("S2-"):
            target_name = s2_name.get(match_id)
            target_addr = s2_addr.get(match_id)
            target_source = "S2"
        else:
            target_name = s3_name.get(match_id)
            target_addr = s3_addr.get(match_id)
            target_source = "S3"

        if pd.isna(target_name):
            continue

        # Name similarity
        name_score = ratio(
            str(source_name).lower(),
            str(target_name).lower()
        )

        # Address similarity only if both exist
        if pd.isna(source_addr) or pd.isna(target_addr):
            address_score = np.nan
        else:
            address_score = ratio(
                str(source_addr).lower(),
                str(target_addr).lower()
            )

        true_rows.append({
            "s1_id": s1_id,
            "target_id": match_id,
            "target_source": target_source,
            "country": country,
            "name_score": name_score,
            "address_score": address_score,
            "address_available": not pd.isna(address_score)
        })


true_pairs_clean = pd.DataFrame(true_rows)

print("\nTRUE PAIR DATASET")
print("=" * 50)
print("Rows:", len(true_pairs_clean))
print("Columns:", true_pairs_clean.columns.tolist())

display(true_pairs_clean.head())

Building true-match pairs: 100%|████████████████████████████████████████████████| 10000/10000 [00:38<00:00, 258.44it/s]



TRUE PAIR DATASET
Rows: 36703
Columns: ['s1_id', 'target_id', 'target_source', 'country', 'name_score', 'address_score', 'address_available']


,s1_id,target_id,target_source,country,name_score,address_score,address_available
0,S1-777210964,S2-12029274,S2,US,97.297297,94.339623,True
1,S1-777210964,S2-407524980,S2,US,100.000000,88.000000,True
2,S1-777210964,S3-970714334,S3,US,95.000000,83.333333,True
3,S1-777210964,S3-712922821,S3,US,77.419355,40.000000,True
4,S1-190133982,S2-523219173,S2,India,80.000000,98.901099,True


In [43]:
# ============================================================
# COUNTRY × TRUE-MATCH SIGNAL DISTRIBUTIONS
# ============================================================

percentiles = [10, 25, 50, 75, 90, 95, 99]

for country in ["US", "India"]:

    subset = true_pairs_clean[
        true_pairs_clean["country"] == country
    ]

    print("\n" + "=" * 65)
    print(f"{country} — TRUE MATCHES")
    print("=" * 65)

    print(f"Pairs: {len(subset):,}")
    print(
        f"Address available: "
        f"{subset['address_available'].mean() * 100:.2f}%"
    )

    print("\nName similarity:")
    for p in percentiles:
        print(
            f"P{p:<2}: "
            f"{subset['name_score'].quantile(p / 100):.2f}"
        )

    print(f"Mean  : {subset['name_score'].mean():.2f}")
    print(f"Median: {subset['name_score'].median():.2f}")

    addr = subset["address_score"].dropna()

    print("\nAddress similarity:")
    for p in percentiles:
        print(
            f"P{p:<2}: "
            f"{addr.quantile(p / 100):.2f}"
        )

    print(f"Mean  : {addr.mean():.2f}")
    print(f"Median: {addr.median():.2f}")


US — TRUE MATCHES
Pairs: 21,865
Address available: 95.21%

Name similarity:
P10: 64.71
P25: 77.19
P50: 88.89
P75: 95.77
P90: 100.00
P95: 100.00
P99: 100.00
Mean  : 84.37
Median: 88.89

Address similarity:
P10: 53.16
P25: 70.97
P50: 84.21
P75: 92.31
P90: 97.67
P95: 100.00
P99: 100.00
Mean  : 80.02
Median: 84.21

India — TRUE MATCHES
Pairs: 14,838
Address available: 96.18%

Name similarity:
P10: 10.00
P25: 57.69
P50: 80.95
P75: 92.59
P90: 97.78
P95: 100.00
P99: 100.00
Mean  : 68.53
Median: 80.95

Address similarity:
P10: 46.67
P25: 65.15
P50: 82.21
P75: 92.12
P90: 97.93
P95: 100.00
P99: 100.00
Mean  : 76.96
Median: 82.21


In [44]:
# ============================================================
# PHASE 1 — BUILD HARD NEGATIVES
# Same country, but NOT a ground-truth match
# ============================================================

import random

RANDOM_SEED = 42
NEG_PER_S1 = 2

rng = np.random.default_rng(RANDOM_SEED)

# ------------------------------------------------------------
# Build GT lookup
# ------------------------------------------------------------

gt_lookup = {}

for _, row in sample.iterrows():
    s1_id = row["source1_entity_id"]

    matches = {
        x.strip()
        for x in str(row["matched_entity_ids"]).split(",")
        if x.strip()
    }

    gt_lookup[s1_id] = matches


# ------------------------------------------------------------
# Candidate pools by country
# Use entity IDs only — much smaller than full DataFrames
# ------------------------------------------------------------

s2_by_country = {
    country: group["entity_id"].to_numpy()
    for country, group in source2.groupby("country")
}

s3_by_country = {
    country: group["entity_id"].to_numpy()
    for country, group in source3.groupby("country")
}


# ------------------------------------------------------------
# Build negative pairs
# ------------------------------------------------------------

negative_rows = []

for _, row in tqdm(
    sample.iterrows(),
    total=len(sample),
    desc="Building hard negatives"
):

    s1_id = row["source1_entity_id"]
    country = s1_country.get(s1_id)

    if pd.isna(country):
        continue

    true_ids = gt_lookup.get(s1_id, set())

    # S2 candidates
    for target_id in s2_by_country.get(country, np.array([])):

        if target_id in true_ids:
            continue

        negative_rows.append({
            "s1_id": s1_id,
            "target_id": target_id,
            "target_source": "S2",
            "country": country,
        })

        if sum(
            1 for x in negative_rows[-NEG_PER_S1:]
            if x["s1_id"] == s1_id
        ) >= NEG_PER_S1:
            break

    # S3 candidates
    if sum(x["s1_id"] == s1_id for x in negative_rows) < NEG_PER_S1:

        for target_id in s3_by_country.get(country, np.array([])):

            if target_id in true_ids:
                continue

            negative_rows.append({
                "s1_id": s1_id,
                "target_id": target_id,
                "target_source": "S3",
                "country": country,
            })

            break


negative_pairs_clean = pd.DataFrame(negative_rows)

print("Negative pairs:", len(negative_pairs_clean))
display(negative_pairs_clean.head())

Building hard negatives: 100%|██████████████████████████████████████████████████| 10000/10000 [00:17<00:00, 573.43it/s]

Negative pairs: 20000


,s1_id,target_id,target_source,country
0,S1-777210964,S2-764573417,S2,US
1,S1-777210964,S2-163963287,S2,US
2,S1-190133982,S2-166376419,S2,India
3,S1-190133982,S2-639257739,S2,India
4,S1-970131671,S2-764573417,S2,US


In [45]:
from rapidfuzz.fuzz import ratio

def add_similarity_scores(df):
    rows = []

    for _, r in df.iterrows():
        s1_id = r["s1_id"]
        target_id = r["target_id"]

        if r["target_source"] == "S2":
            t_name = s2_name.get(target_id)
            t_addr = s2_addr.get(target_id)
        else:
            t_name = s3_name.get(target_id)
            t_addr = s3_addr.get(target_id)

        src_name = s1_name.get(s1_id)
        src_addr = s1_addr.get(s1_id)

        name_score = ratio(str(src_name).lower(), str(t_name).lower())

        if pd.isna(src_addr) or pd.isna(t_addr):
            addr_score = np.nan
        else:
            addr_score = ratio(
                str(src_addr).lower(),
                str(t_addr).lower()
            )

        rows.append({
            **r.to_dict(),
            "name_score": name_score,
            "address_score": addr_score
        })

    return pd.DataFrame(rows)


negative_scored = add_similarity_scores(negative_pairs_clean)

print("Negative pairs:", len(negative_scored))
display(negative_scored.head())

Negative pairs: 20000


,s1_id,target_id,target_source,country,name_score,address_score
0,S1-777210964,S2-764573417,S2,US,29.787234,45.283019
1,S1-777210964,S2-163963287,S2,US,41.379310,27.692308
2,S1-190133982,S2-166376419,S2,India,8.000000,32.835821
3,S1-190133982,S2-639257739,S2,India,9.302326,37.681159
4,S1-970131671,S2-764573417,S2,US,37.735849,36.363636


In [46]:
for country in ["US", "India"]:
    x = negative_scored[negative_scored["country"] == country]

    print(f"\n{country} — HARD NEGATIVES")
    print(f"Pairs: {len(x):,}")

    for col in ["name_score", "address_score"]:
        print(f"\n{col}")
        print(x[col].describe(
            percentiles=[.10, .25, .50, .75, .90, .95, .99]
        ).round(2))


US — HARD NEGATIVES
Pairs: 12,000

name_score
count    12000.00
mean        29.41
std          6.95
min          0.00
10%         21.05
25%         25.00
50%         29.17
75%         33.33
90%         37.50
95%         40.91
99%         48.79
max         80.00
Name: name_score, dtype: float64

address_score
count    12000.00
mean        34.96
std          6.59
min         17.86
10%         27.27
25%         30.14
50%         33.96
75%         39.29
90%         44.12
95%         47.06
99%         52.00
max         75.00
Name: address_score, dtype: float64

India — HARD NEGATIVES
Pairs: 8,000

name_score
count    8000.00
mean        8.32
std         1.75
min         3.45
10%         6.15
25%         7.27
50%         8.33
75%         9.68
90%        10.53
95%        10.91
99%        12.00
max        13.64
Name: name_score, dtype: float64

address_score
count    8000.00
mean       35.46
std         7.14
min        19.18
10%        28.19
25%        30.88
50%        34.09
75%        38.19


In [47]:
for country in ["US", "India"]:
    pos = true_pairs_clean[true_pairs_clean["country"] == country]
    neg = negative_scored[negative_scored["country"] == country]

    print(f"\n===== {country} =====")

    for threshold in [50, 60, 70, 80, 90]:
        pos_name = (pos["name_score"] >= threshold).mean() * 100
        neg_name = (neg["name_score"] >= threshold).mean() * 100

        pos_addr = (pos["address_score"] >= threshold).mean() * 100
        neg_addr = (neg["address_score"] >= threshold).mean() * 100

        print(
            f"T={threshold}: "
            f"Name true={pos_name:.1f}% | neg={neg_name:.1f}% | "
            f"Address true={pos_addr:.1f}% | neg={neg_addr:.1f}%"
        )


===== US =====
T=50: Name true=96.8% | neg=1.0% | Address true=89.1% | neg=2.4%
T=60: Name true=93.4% | neg=0.2% | Address true=80.5% | neg=0.1%
T=70: Name true=84.9% | neg=0.0% | Address true=72.4% | neg=0.0%
T=80: Name true=71.0% | neg=0.0% | Address true=58.4% | neg=0.0%
T=90: Name true=46.3% | neg=0.0% | Address true=30.6% | neg=0.0%

===== India =====
T=50: Name true=78.2% | neg=0.0% | Address true=84.7% | neg=5.3%
T=60: Name true=73.5% | neg=0.0% | Address true=76.8% | neg=1.1%
T=70: Name true=64.7% | neg=0.0% | Address true=66.9% | neg=0.1%
T=80: Name true=51.8% | neg=0.0% | Address true=52.5% | neg=0.0%
T=90: Name true=32.0% | neg=0.0% | Address true=30.2% | neg=0.0%


In [48]:
for country in ["US", "India"]:
    pos = true_pairs_clean[true_pairs_clean["country"] == country].copy()
    neg = negative_scored[negative_scored["country"] == country].copy()

    for df in [pos, neg]:
        df["combined"] = (
            0.5 * df["name_score"] +
            0.5 * df["address_score"]
        )

    print(f"\n===== {country} =====")

    for t in [40, 50, 60, 70, 80, 90]:
        tp = (pos["combined"] >= t).mean() * 100
        fp = (neg["combined"] >= t).mean() * 100

        print(
            f"T={t}: true={tp:.1f}% | hard-negative={fp:.1f}%"
        )


===== US =====
T=40: true=94.9% | hard-negative=5.1%
T=50: true=94.3% | hard-negative=0.1%
T=60: true=91.2% | hard-negative=0.0%
T=70: true=82.3% | hard-negative=0.0%
T=80: true=60.6% | hard-negative=0.0%
T=90: true=25.5% | hard-negative=0.0%

===== India =====
T=40: true=88.9% | hard-negative=0.1%
T=50: true=81.3% | hard-negative=0.0%
T=60: true=71.7% | hard-negative=0.0%
T=70: true=60.8% | hard-negative=0.0%
T=80: true=42.9% | hard-negative=0.0%
T=90: true=17.3% | hard-negative=0.0%


In [49]:
for country in ["US", "India"]:
    pos = true_pairs_clean[true_pairs_clean["country"] == country]
    neg = negative_scored[negative_scored["country"] == country]

    print(f"\n===== {country} =====")

    for name_w in [0.3, 0.5, 0.7]:
        addr_w = 1 - name_w

        pos_score = (
            name_w * pos["name_score"] +
            addr_w * pos["address_score"]
        )

        neg_score = (
            name_w * neg["name_score"] +
            addr_w * neg["address_score"]
        )

        print(
            f"Name={name_w:.1f}, Address={addr_w:.1f} | "
            f"T50 true={((pos_score >= 50).mean()*100):.1f}% | "
            f"negative={((neg_score >= 50).mean()*100):.1f}%"
        )


===== US =====
Name=0.3, Address=0.7 | T50 true=94.4% | negative=0.1%
Name=0.5, Address=0.5 | T50 true=94.3% | negative=0.1%
Name=0.7, Address=0.3 | T50 true=93.2% | negative=0.2%

===== India =====
Name=0.3, Address=0.7 | T50 true=86.3% | negative=0.1%
Name=0.5, Address=0.5 | T50 true=81.3% | negative=0.0%
Name=0.7, Address=0.3 | T50 true=75.5% | negative=0.0%


In [50]:
for country in ["US", "India"]:
    x = true_pairs_clean[true_pairs_clean["country"] == country]

    print(f"\n{country}")

    print("Name < 50, Address >= 70:",
          ((x.name_score < 50) & (x.address_score >= 70)).mean() * 100)

    print("Name >= 70, Address < 50:",
          ((x.name_score >= 70) & (x.address_score < 50)).mean() * 100)

    print("Both >= 70:",
          ((x.name_score >= 70) & (x.address_score >= 70)).mean() * 100)

    print("Both < 50:",
          ((x.name_score < 50) & (x.address_score < 50)).mean() * 100)


US
Name < 50, Address >= 70: 2.4376857992225016
Name >= 70, Address < 50: 5.168076835124628
Both >= 70: 61.3857763549051
Both < 50: 0.16464669563228906

India
Name < 50, Address >= 70: 13.579997304218896
Name >= 70, Address < 50: 6.759671114705486
Both >= 70: 44.16363391292627
Both < 50: 3.140584984499259


In [51]:
for country in ["US", "India"]:
    x = true_pairs_clean[true_pairs_clean["country"] == country]

    print(f"\n===== {country} =====")

    print(
        x.groupby("target_source")[["name_score", "address_score"]]
        .agg(["mean", "median"])
        .round(2)
    )


===== US =====
              name_score        address_score       
                    mean median          mean median
target_source                                       
S2                 85.03  88.89         82.44  87.18
S3                 83.75  88.24         77.76  82.14

===== India =====
              name_score        address_score       
                    mean median          mean median
target_source                                       
S2                 65.74  80.00         82.29  86.25
S3                 71.11  81.82         72.03  76.77


In [52]:
for country in ["US", "India"]:
    print(f"\n===== {country} =====")

    for source in ["S2", "S3"]:
        x = true_pairs_clean[
            (true_pairs_clean["country"] == country) &
            (true_pairs_clean["target_source"] == source)
        ]

        print(f"\n{source}")
        print(
            x[["name_score", "address_score"]]
            .quantile([.10, .25, .50, .75, .90])
            .round(2)
        )


===== US =====

S2
      name_score  address_score
0.10       66.67          54.55
0.25       78.43          73.53
0.50       88.89          87.18
0.75       96.00          95.52
0.90      100.00         100.00

S3
      name_score  address_score
0.10       63.64          52.17
0.25       75.86          68.75
0.50       88.24          82.14
0.75       95.65          89.16
0.90      100.00          93.33

===== India =====

S2
      name_score  address_score
0.10        9.38          59.02
0.25       47.06          74.29
0.50       80.00          86.25
0.75       92.31          94.03
0.90       97.67         100.00

S3
      name_score  address_score
0.10       11.28          40.98
0.25       61.54          56.25
0.50       81.82          76.77
0.75       92.72          89.43
0.90       97.87          95.18


In [54]:
true_pairs_clean.groupby(
    ["country", "target_source"]
)["address_available"].mean().mul(100).round(2)

country  target_source
India    S2               96.20
         S3               96.16
US       S2               95.05
         S3               95.35
Name: address_available, dtype: float64

In [55]:
print(source1.columns.tolist())

['entity_id', 'business_name', 'business_address', 'country']


# Phase 1 — Exploratory Data Analysis (EDA)

## 1. Dataset Overview

The entity-matching dataset contains three source datasets and one ground-truth mapping.

| Dataset | Rows | Columns |
|---|---:|---:|
| Source 1 (S1) | 2,206,821 | 4 |
| Source 2 (S2) | 5,034,616 | 4 |
| Source 3 (S3) | 5,285,603 | 4 |
| Ground Truth | 2,206,821 | 2 |

### Source schema

All three sources contain:

- `entity_id`
- `business_name`
- `business_address`
- `country`

### Ground-truth schema

- `source1_entity_id`
- `matched_entity_ids`

A Source 1 entity can map to zero, one, or multiple entities in S2/S3. Therefore, the task is not simple one-to-one deduplication; it is a **one-to-many entity resolution / entity matching problem**.

---

# 2. Country Distribution

The datasets contain two countries:

| Source | US | India |
|---|---:|---:|
| S1 | 1,323,633 | 883,188 |
| S2 | 3,016,817 | 2,017,799 |
| S3 | 3,170,056 | 2,115,547 |

The overall distribution is approximately 60% US and 40% India.

This makes `country` a useful contextual variable and a potential blocking constraint.

---

# 3. Ground-Truth Structure

The ground truth contains:

- 2,206,821 Source 1 entities
- 123,247 S1 entities with zero matches
- Multiple S2/S3 matches for many S1 entities
- Maximum observed number of matches for one S1 entity: 11

Therefore, the matching system must support **multiple valid matches for a single source entity**.

---

# 4. Address Availability

The address field is highly populated.

| Source | Overall usable address |
|---|---:|
| S1 | 100.00% |
| S2 | 96.64% |
| S3 | 96.67% |

Country-level availability:

| Source | US | India |
|---|---:|---:|
| S1 | 100.00% | 100.00% |
| S2 | 96.32% | 97.13% |
| S3 | 96.50% | 96.93% |

### Observation

Address missingness is low and does not show a major country-specific difference.

Therefore:

- Address should **not** be discarded because of missingness.
- Missing address should cause the address signal to be ignored for that pair rather than automatically rejecting the match.

---

# 5. True-Match Signal Analysis

A clean true-match dataset was reconstructed from the ground truth.

Total true pairs analyzed:

**36,703**

### True-match address availability

| Country | Pairs | Address available |
|---|---:|---:|
| US | 21,865 | 95.21% |
| India | 14,838 | 96.18% |

---

# 6. Country-Specific Signal Behavior

## 6.1 US True Matches

### Business Name

| Percentile | Similarity |
|---|---:|
| P10 | 64.71 |
| P25 | 77.19 |
| P50 | 88.89 |
| P75 | 95.77 |
| P90 | 100.00 |
| Mean | 84.37 |

### Business Address

| Percentile | Similarity |
|---|---:|
| P10 | 53.16 |
| P25 | 70.97 |
| P50 | 84.21 |
| P75 | 92.31 |
| P90 | 97.67 |
| Mean | 80.02 |

---

## 6.2 India True Matches

### Business Name

| Percentile | Similarity |
|---|---:|
| P10 | 10.00 |
| P25 | 57.69 |
| P50 | 80.95 |
| P75 | 92.59 |
| P90 | 97.78 |
| Mean | 68.53 |

### Business Address

| Percentile | Similarity |
|---|---:|
| P10 | 46.67 |
| P25 | 65.15 |
| P50 | 82.21 |
| P75 | 92.12 |
| P90 | 97.93 |
| Mean | 76.96 |

---

# 7. Main Country-Specific Finding

Business-name similarity differs substantially between the US and India.

| Signal | US mean | India mean |
|---|---:|---:|
| Name | 84.37 | 68.53 |
| Address | 80.02 | 76.96 |

The name signal has a much larger country difference than the address signal.

The India P10 for name similarity is only **10**, showing that legitimate Indian matches can have extremely low name similarity.

In contrast, address similarity is considerably more stable between the two countries.

### Current interpretation

`business_name` should not be treated as having identical reliability across countries.

`business_address` appears to be a more stable cross-country signal.

---

# 8. Hard-Negative Analysis

20,000 same-country hard-negative pairs were generated.

These are candidates that:

- belong to the same country as the S1 entity
- are not present in the ground truth for that S1 entity

## US Hard Negatives

### Name

- Mean: 29.41
- Median: 29.17
- P90: 37.50
- P95: 40.91
- P99: 48.79

### Address

- Mean: 34.96
- Median: 33.96
- P90: 44.12
- P95: 47.06
- P99: 52.00

## India Hard Negatives

### Name

- Mean: 8.32
- Median: 8.33
- P90: 10.53
- P95: 10.91
- P99: 12.00

### Address

- Mean: 35.46
- Median: 34.09
- P90: 43.90
- P95: 50.37
- P99: 60.38

---

# 9. Single-Signal Threshold Diagnostics

These results are exploratory only and are **not final thresholds**.

At similarity threshold 60:

| Country | Signal | True retained | Hard negatives |
|---|---|---:|---:|
| US | Name | 93.4% | 0.2% |
| US | Address | 80.5% | 0.1% |
| India | Name | 73.5% | 0.0% |
| India | Address | 76.8% | 1.1% |

### Observation

Name is particularly strong for US matches.

For India, address

# Text Normalization Analysis

In [58]:
for name, df in [("S1", source1), ("S2", source2), ("S3", source3)]:
    print(f"\n{name} columns:")
    print(df.columns.tolist())


S1 columns:
['entity_id', 'business_name', 'business_address', 'country']

S2 columns:
['entity_id', 'business_name', 'business_address', 'country']

S3 columns:
['entity_id', 'business_name', 'business_address', 'country']


In [59]:
import re
import unicodedata
import pandas as pd


# ============================================================
# 1. NORMALIZATION FUNCTION
# ============================================================

def normalize_text(x):
    if pd.isna(x):
        return ""
    
    x = str(x)
    x = unicodedata.normalize("NFKC", x)
    x = x.lower()
    
    # Replace punctuation/special characters with spaces
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    
    # Normalize whitespace
    x = re.sub(r"\s+", " ", x).strip()
    
    return x


# ============================================================
# 2. ANALYZE NORMALIZATION EFFECT
# ============================================================

def analyze_normalization(df, source_name, column, sample_size=10000):
    
    # Only access the required column
    s = df[column].dropna()
    
    n = min(sample_size, len(s))
    
    sample = s.sample(n=n, random_state=42)
    normalized = sample.map(normalize_text)
    
    changed = sample.astype(str) != normalized
    
    raw_unique = sample.nunique()
    normalized_unique = normalized.nunique()
    
    print(f"\n{'='*60}")
    print(f"{source_name} → {column}")
    print(f"{'='*60}")
    
    print(f"Sample size          : {n:,}")
    print(f"Changed              : {changed.sum():,}")
    print(f"Change %             : {changed.mean()*100:.2f}%")
    print(f"Raw unique           : {raw_unique:,}")
    print(f"Normalized unique   : {normalized_unique:,}")
    
    # Collision count
    collision_groups = (
        pd.DataFrame({
            "raw": sample.astype(str).values,
            "normalized": normalized.values
        })
        .groupby("normalized")["raw"]
        .nunique()
    )
    
    collisions = (collision_groups > 1).sum()
    
    print(f"Normalization collisions: {collisions:,}")
    
    return pd.DataFrame({
        "raw": sample.astype(str).values,
        "normalized": normalized.values
    })


# ============================================================
# 3. BUSINESS NAME
# ============================================================

s1_name = analyze_normalization(
    source1, "S1", "business_name"
)

s2_name = analyze_normalization(
    source2, "S2", "business_name"
)

s3_name = analyze_normalization(
    source3, "S3", "business_name"
)


# ============================================================
# 4. BUSINESS ADDRESS
# ============================================================

s1_address = analyze_normalization(
    source1, "S1", "business_address"
)

s2_address = analyze_normalization(
    source2, "S2", "business_address"
)

s3_address = analyze_normalization(
    source3, "S3", "business_address"
)


S1 → business_name
Sample size          : 10,000
Changed              : 10,000
Change %             : 100.00%
Raw unique           : 9,842
Normalized unique   : 9,824
Normalization collisions: 18

S2 → business_name
Sample size          : 10,000
Changed              : 9,767
Change %             : 97.67%
Raw unique           : 9,982
Normalized unique   : 9,068
Normalization collisions: 25

S3 → business_name
Sample size          : 10,000
Changed              : 9,773
Change %             : 97.73%
Raw unique           : 9,982
Normalized unique   : 9,498
Normalization collisions: 22

S1 → business_address
Sample size          : 10,000
Changed              : 10,000
Change %             : 100.00%
Raw unique           : 9,998
Normalized unique   : 9,998
Normalization collisions: 0

S2 → business_address
Sample size          : 10,000
Changed              : 10,000
Change %             : 100.00%
Raw unique           : 9,999
Normalized unique   : 9,999
Normalization collisions: 0

S3 → business_

In [60]:
print("\nS1 BUSINESS NAME EXAMPLES")
display(
    s1_name[s1_name["raw"] != s1_name["normalized"]]
    .head(30)
)


S1 BUSINESS NAME EXAMPLES


,raw,normalized
0,Pediatric Medicine PLLC,pediatric medicine pllc
1,Fetech National Twin,fetech national twin
2,General Design Innovations LLC,general design innovations llc
3,Construction Ideaz Papers Private Limited,construction ideaz papers private limited
4,Penaloza and Bittle First Inc.,penaloza and bittle first inc
5,National Investments LLC,national investments llc
6,Reus Idaho Company,reus idaho company
7,Creative Projects Limited,creative projects limited
8,Goar Stone LLC,goar stone llc
9,Lakshmi Consultants Private Limited,lakshmi consultants private limited


In [61]:
collision_examples = (
    s1_name
    .groupby("normalized")
    .filter(lambda x: len(x) > 1)
    .sort_values("normalized")
)

display(collision_examples.head(50))

,raw,normalized
548,20/20 Optical,20 20 optical
9428,20/20 Optical,20 20 optical
8234,Ace Business Private Limited,ace business private limited
7292,Ace Business Private Limited,ace business private limited
3511,Aditya It Private Limited,aditya it private limited
3262,Aditya It Private Limited,aditya it private limited
5877,Al It Private Limited,al it private limited
9520,Al It Private Limited,al it private limited
9237,Anderson's First Painting,anderson s first painting
7678,Anderson's First Painting,anderson s first painting


In [62]:
import re
import pandas as pd

COMPANY_SUFFIXES = [
    r"\bprivate limited\b",
    r"\bpvt ltd\b",
    r"\bpvt limited\b",
    r"\bpublic limited\b",
    r"\bllc\b",
    r"\bincorporated\b",
    r"\binc\b",
    r"\blimited\b",
    r"\bltd\b",
    r"\bcorp\b",
    r"\bcorporation\b",
    r"\bco\b",
    r"\bcompany\b",
]

suffix_pattern = re.compile("|".join(COMPANY_SUFFIXES), flags=re.IGNORECASE)


def remove_company_suffix(text):
    if pd.isna(text):
        return ""
    
    text = str(text).lower()
    text = suffix_pattern.sub(" ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    
    return text


def suffix_analysis(df, source_name, sample_size=10000):
    s = df["business_name"].dropna()
    sample = s.sample(min(sample_size, len(s)), random_state=42)
    
    normalized = sample.map(remove_company_suffix)
    
    print(f"\n{'='*60}")
    print(source_name)
    print(f"{'='*60}")
    print("Sample:", len(sample))
    print("Raw unique:", sample.nunique())
    print("After suffix removal:", normalized.nunique())
    print(
        "Unique reduction:",
        round((1 - normalized.nunique() / sample.nunique()) * 100, 2),
        "%"
    )
    
    return pd.DataFrame({
        "raw": sample.values,
        "normalized": normalized.values
    })


s1_suffix = suffix_analysis(source1, "S1")
s2_suffix = suffix_analysis(source2, "S2")
s3_suffix = suffix_analysis(source3, "S3")


S1
Sample: 10000
Raw unique: 9842
After suffix removal: 9655
Unique reduction: 1.9 %

S2
Sample: 10000
Raw unique: 9982
After suffix removal: 9004
Unique reduction: 9.8 %

S3
Sample: 10000
Raw unique: 9982
After suffix removal: 9443
Unique reduction: 5.4 %


In [63]:
display(
    s1_suffix[
        s1_suffix["raw"] != s1_suffix["normalized"]
    ].head(50)
)

,raw,normalized
0,Pediatric Medicine PLLC,pediatric medicine pllc
1,Fetech National Twin,fetech national twin
2,General Design Innovations LLC,general design innovations
3,Construction Ideaz Papers Private Limited,construction ideaz papers
4,Penaloza and Bittle First Inc.,penaloza and bittle first
5,National Investments LLC,national investments
6,Reus Idaho Company,reus idaho
7,Creative Projects Limited,creative projects
8,Goar Stone LLC,goar stone
9,Lakshmi Consultants Private Limited,lakshmi consultants


In [64]:
collisions = (
    s1_suffix
    .groupby("normalized")
    .filter(lambda x: len(x) > 1)
    .sort_values("normalized")
)

display(collisions.head(50))

,raw,normalized
548,20/20 Optical,20 20 optical
9428,20/20 Optical,20 20 optical
7980,24HR Fitness Corporation,24hr fitness
4479,24HR Fitness Inc,24hr fitness
1169,24HR Locksmith LLC,24hr locksmith
5582,24HR Locksmith Corp,24hr locksmith
8234,Ace Business Private Limited,ace business
7292,Ace Business Private Limited,ace business
3262,Aditya It Private Limited,aditya it
3511,Aditya It Private Limited,aditya it


In [65]:
import re
from collections import Counter

suffixes = [
    "private limited",
    "pvt ltd",
    "pvt limited",
    "public limited",
    "llc",
    "incorporated",
    "inc",
    "limited",
    "ltd",
    "corp",
    "corporation",
    "co",
    "company",
    "llp",
]

suffix_pattern = re.compile(
    r"\b(" + "|".join(map(re.escape, suffixes)) + r")\b",
    flags=re.IGNORECASE
)


def count_suffixes(df, source_name):
    counts = Counter()
    
    # Process only the required column
    for value in df["business_name"].dropna():
        matches = suffix_pattern.findall(str(value).lower())
        counts.update(matches)
    
    result = pd.DataFrame(
        counts.items(),
        columns=["suffix", "count"]
    ).sort_values("count", ascending=False)
    
    result["percentage"] = (
        result["count"] / df["business_name"].notna().sum() * 100
    )
    
    print(f"\n{'='*50}")
    print(source_name)
    print(f"{'='*50}")
    display(result)
    
    return result


s1_suffix_counts = count_suffixes(source1, "S1")
s2_suffix_counts = count_suffixes(source2, "S2")
s3_suffix_counts = count_suffixes(source3, "S3")


S1


,suffix,count,percentage
2,private limited,431872,19.569870
1,llc,355736,16.119839
0,inc,238309,10.798746
9,pvt ltd,94922,4.301300
3,limited,85972,3.895740
4,ltd,53677,2.432322
7,llp,39726,1.800146
8,corp,34792,1.576566
6,co,27692,1.254837
10,corporation,14372,0.651254



S2


,suffix,count,percentage
3,llc,532379,10.574376
0,inc,403240,8.009353
2,ltd,326198,6.479106
4,limited,296406,5.887363
5,private limited,204337,4.058643
11,corp,142833,2.837020
8,co,140870,2.798030
10,pvt ltd,74935,1.488396
7,llp,65179,1.294618
1,corporation,63182,1.254952



S3


,suffix,count,percentage
1,llc,575634,10.890629
2,inc,422989,8.002683
6,limited,361628,6.841772
0,ltd,333231,6.304519
4,private limited,283737,5.368124
5,co,145168,2.746486
9,corp,141308,2.673457
8,pvt ltd,97472,1.844108
12,llp,77774,1.471435
7,corporation,58592,1.108523


In [69]:
# ============================================================
# FAST + VISIBLE PROGRESS GT NORMALIZATION ANALYSIS
# ============================================================

import ast
import re
import pandas as pd
from rapidfuzz.fuzz import ratio


# ------------------------------------------------------------
# 1. NORMALIZATION
# ------------------------------------------------------------

def basic_norm(x):
    if pd.isna(x):
        return ""
    x = str(x).lower()
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    return re.sub(r"\s+", " ", x).strip()


def suffix_norm(x):
    x = basic_norm(x)

    x = re.sub(
        r"\b(private limited|pvt limited|pvt ltd|public limited|"
        r"limited|ltd|llc|incorporated|inc|corporation|corp|llp)\b",
        " ",
        x
    )

    return re.sub(r"\s+", " ", x).strip()


# ------------------------------------------------------------
# 2. ENTITY LOOKUPS
# ------------------------------------------------------------

print("Creating entity indexes...")

s1_names = source1.set_index("entity_id")["business_name"].to_dict()
s2_names = source2.set_index("entity_id")["business_name"].to_dict()
s3_names = source3.set_index("entity_id")["business_name"].to_dict()

print("✓ Entity indexes ready")


# ------------------------------------------------------------
# 3. SAMPLE GT
# ------------------------------------------------------------

N = min(25_000, len(gt))

print(f"\nSampling {N:,} GT rows...")

gt_sample = gt[
    ["source1_entity_id", "matched_entity_ids"]
].sample(
    N,
    random_state=42
).copy()

print("✓ GT sample ready")


# ------------------------------------------------------------
# 4. BUILD ONLY REQUIRED NAME CACHE
# ------------------------------------------------------------

print("\nBuilding name cache...")

required_s1 = set(gt_sample["source1_entity_id"])

required_s1_names = {
    x: s1_names[x]
    for x in required_s1
    if x in s1_names
}

required_targets = set()

for x in gt_sample["matched_entity_ids"]:
    try:
        ids = x if isinstance(x, list) else ast.literal_eval(str(x))
        if isinstance(ids, list):
            required_targets.update(map(str, ids))
    except:
        pass

required_target_names = {}

for entity_id in required_targets:

    if entity_id in s2_names:
        required_target_names[entity_id] = s2_names[entity_id]

    elif entity_id in s3_names:
        required_target_names[entity_id] = s3_names[entity_id]


print(
    f"✓ Required S1 names    : {len(required_s1_names):,}"
)

print(
    f"✓ Required target names: {len(required_target_names):,}"
)


# ------------------------------------------------------------
# 5. NORMALIZE ONLY REQUIRED NAMES
# ------------------------------------------------------------

print("\nNormalizing names...")

name_cache = {}

all_required_names = list(
    required_s1_names.values()
) + list(
    required_target_names.values()
)

# Deduplicate before normalization
all_required_names = set(
    x for x in all_required_names
    if not pd.isna(x)
)

for i, name in enumerate(all_required_names, 1):

    name_cache[name] = (
        basic_norm(name),
        suffix_norm(name)
    )

    if i % 5000 == 0:
        print(
            f"  Normalized {i:,}/{len(all_required_names):,}"
        )

print("✓ Normalization complete")


# ------------------------------------------------------------
# 6. PARSE + SCORE WITH PROGRESS
# ------------------------------------------------------------

print("\nStarting GT comparison...")

raw_scores = []
suffix_scores = []

total_rows = len(gt_sample)

for row_num, (_, row) in enumerate(
    gt_sample.iterrows(),
    1
):

    s1_name = required_s1_names.get(
        row["source1_entity_id"]
    )

    if s1_name is None:
        continue

    s1_raw, s1_suffix = name_cache[s1_name]

    try:
        matched_ids = row["matched_entity_ids"]

        if not isinstance(matched_ids, list):
            matched_ids = ast.literal_eval(
                str(matched_ids)
            )

        if not isinstance(matched_ids, list):
            continue

    except:
        continue

    for matched_id in matched_ids:

        matched_id = str(matched_id)

        target_name = required_target_names.get(
            matched_id
        )

        if target_name is None:
            continue

        target_raw, target_suffix = name_cache[
            target_name
        ]

        raw_scores.append(
            ratio(s1_raw, target_raw)
        )

        suffix_scores.append(
            ratio(s1_suffix, target_suffix)
        )

    # Visible progress
    if row_num % 1000 == 0:

        print(
            f"Progress: {row_num:,}/{total_rows:,} "
            f"({row_num / total_rows * 100:.1f}%) | "
            f"Pairs: {len(raw_scores):,}"
        )


# ------------------------------------------------------------
# 7. RESULTS
# ------------------------------------------------------------

scores = pd.DataFrame({
    "raw_score": raw_scores,
    "suffix_score": suffix_scores
})

scores["improvement"] = (
    scores["suffix_score"] -
    scores["raw_score"]
)

print("\n" + "=" * 60)
print("FINAL RESULTS")
print("=" * 60)

print(f"GT rows processed : {total_rows:,}")
print(f"GT pairs analyzed  : {len(scores):,}")

print("\nMean:")
print(f"Raw    : {scores.raw_score.mean():.2f}")
print(f"Suffix : {scores.suffix_score.mean():.2f}")

print("\nMedian:")
print(f"Raw    : {scores.raw_score.median():.2f}")
print(f"Suffix : {scores.suffix_score.median():.2f}")

print("\nChange:")
print(
    f"Improved : {(scores.improvement > 0).mean()*100:.2f}%"
)

print(
    f"Same     : {(scores.improvement == 0).mean()*100:.2f}%"
)

print(
    f"Worse    : {(scores.improvement < 0).mean()*100:.2f}%"
)

print("\nSimilarity >= 90:")

print(
    f"Raw    : {(scores.raw_score >= 90).mean()*100:.2f}%"
)

print(
    f"Suffix : {(scores.suffix_score >= 90).mean()*100:.2f}%"
)

print("\nImprovement statistics:")
print(scores["improvement"].describe())

Creating entity indexes...
✓ Entity indexes ready

Sampling 25,000 GT rows...
✓ GT sample ready

Building name cache...
✓ Required S1 names    : 25,000
✓ Required target names: 0

Normalizing names...
  Normalized 5,000/24,143
  Normalized 10,000/24,143
  Normalized 15,000/24,143
  Normalized 20,000/24,143
✓ Normalization complete

Starting GT comparison...

FINAL RESULTS
GT rows processed : 25,000
GT pairs analyzed  : 0

Mean:
Raw    : nan
Suffix : nan

Median:
Raw    : nan
Suffix : nan

Change:
Improved : nan%
Same     : nan%
Worse    : nan%

Similarity >= 90:
Raw    : nan%
Suffix : nan%

Improvement statistics:
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: improvement, dtype: float64


In [70]:
print(gt.columns.tolist())

display(gt.head(10))

print("\nGT dtypes:")
print(gt.dtypes)

print("\nExample matched_entity_ids:")
for x in gt["matched_entity_ids"].head(10):
    print(repr(x), type(x))

['source1_entity_id', 'matched_entity_ids']


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."
5,S1-18727616,"S2-755677256,S3-187831601,S3-641489370,S3-4762..."
6,S1-318373630,"S2-660036492,S3-804600254"
7,S1-86989137,"S3-274817120,S3-312496301"
8,S1-29845983,"S2-648035184,S3-588502663"
9,S1-789009573,"S2-383871912,S3-74481402,S3-576451439"



GT dtypes:
source1_entity_id     object
matched_entity_ids    object
dtype: object

Example matched_entity_ids:
'S2-681193310,S2-743505751,S3-775321672,S3-11291185,S3-860443364' <class 'str'>
'S2-249013014,S2-197070651,S3-478195123,S3-384364074' <class 'str'>
'S2-790675320,S2-479876582,S3-878454467' <class 'str'>
'S2-153058913,S2-24659151,S3-679606215' <class 'str'>
'S2-478959098,S2-553508714,S2-625774905,S3-728090388,S3-928796641,S3-449308785' <class 'str'>
'S2-755677256,S3-187831601,S3-641489370,S3-476250621' <class 'str'>
'S2-660036492,S3-804600254' <class 'str'>
'S3-274817120,S3-312496301' <class 'str'>
'S2-648035184,S3-588502663' <class 'str'>
'S2-383871912,S3-74481402,S3-576451439' <class 'str'>


In [71]:
# ============================================================
# FAST GT SUFFIX-NORMALIZATION ANALYSIS
# CORRECT GT FORMAT
# ============================================================

import re
import pandas as pd
from rapidfuzz.fuzz import ratio


# ------------------------------------------------------------
# 1. NORMALIZATION
# ------------------------------------------------------------

def basic_norm(x):
    if pd.isna(x):
        return ""

    x = str(x).lower()
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    return re.sub(r"\s+", " ", x).strip()


def suffix_norm(x):
    x = basic_norm(x)

    x = re.sub(
        r"\b(private limited|pvt limited|pvt ltd|public limited|"
        r"limited|ltd|llc|incorporated|inc|corporation|corp|llp)\b",
        " ",
        x
    )

    return re.sub(r"\s+", " ", x).strip()


# ------------------------------------------------------------
# 2. ENTITY LOOKUPS
# ------------------------------------------------------------

print("Creating indexes...")

s1_names = dict(
    zip(source1["entity_id"].astype(str), source1["business_name"])
)

s2_names = dict(
    zip(source2["entity_id"].astype(str), source2["business_name"])
)

s3_names = dict(
    zip(source3["entity_id"].astype(str), source3["business_name"])
)

print("✓ Indexes ready")


# ------------------------------------------------------------
# 3. SAMPLE GT
# ------------------------------------------------------------

N = min(25_000, len(gt))

gt_sample = gt[
    ["source1_entity_id", "matched_entity_ids"]
].sample(
    N,
    random_state=42
).copy()

print(f"✓ GT sample: {len(gt_sample):,}")


# ------------------------------------------------------------
# 4. COLLECT ONLY REQUIRED IDS
# ------------------------------------------------------------

print("\nCollecting required entities...")

required_s1 = set(
    gt_sample["source1_entity_id"].astype(str)
)

required_s2 = set()
required_s3 = set()

for value in gt_sample["matched_entity_ids"]:

    if pd.isna(value):
        continue

    for entity in str(value).split(","):

        entity = entity.strip()

        if entity.startswith("S2-"):
            required_s2.add(entity[3:])

        elif entity.startswith("S3-"):
            required_s3.add(entity[3:])


print(f"Required S1: {len(required_s1):,}")
print(f"Required S2: {len(required_s2):,}")
print(f"Required S3: {len(required_s3):,}")


# ------------------------------------------------------------
# 5. NORMALIZE ONLY REQUIRED NAMES
# ------------------------------------------------------------

print("\nBuilding normalization cache...")

name_cache = {}

required_names = []

for entity_id in required_s1:

    name = s1_names.get(entity_id)

    if name is not None:
        required_names.append(name)


for entity_id in required_s2:

    name = s2_names.get(entity_id)

    if name is not None:
        required_names.append(name)


for entity_id in required_s3:

    name = s3_names.get(entity_id)

    if name is not None:
        required_names.append(name)


# Unique names only
required_names = set(
    x for x in required_names
    if not pd.isna(x)
)

print(f"Unique names to normalize: {len(required_names):,}")


for i, name in enumerate(required_names, 1):

    name_cache[name] = (
        basic_norm(name),
        suffix_norm(name)
    )

    if i % 5000 == 0:
        print(
            f"Normalized {i:,}/{len(required_names):,}"
        )

print("✓ Normalization complete")


# ------------------------------------------------------------
# 6. GT MATCH ANALYSIS
# ------------------------------------------------------------

print("\nStarting GT comparison...")

raw_scores = []
suffix_scores = []

total_rows = len(gt_sample)

for row_num, (_, row) in enumerate(
    gt_sample.iterrows(),
    1
):

    s1_id = str(row["source1_entity_id"])

    s1_name = s1_names.get(s1_id)

    if s1_name is None:
        continue

    s1_raw, s1_suffix = name_cache[s1_name]


    # GT matched IDs
    matched_ids = str(
        row["matched_entity_ids"]
    ).split(",")


    for matched_id in matched_ids:

        matched_id = matched_id.strip()

        if not matched_id:
            continue


        # Correct S2/S3 routing
        if matched_id.startswith("S2-"):

            target_id = matched_id[3:]
            target_name = s2_names.get(target_id)

        elif matched_id.startswith("S3-"):

            target_id = matched_id[3:]
            target_name = s3_names.get(target_id)

        else:
            continue


        if target_name is None:
            continue


        target_raw, target_suffix = name_cache[
            target_name
        ]


        raw_scores.append(
            ratio(s1_raw, target_raw)
        )

        suffix_scores.append(
            ratio(s1_suffix, target_suffix)
        )


    # Progress
    if row_num % 1000 == 0:

        print(
            f"Progress: {row_num:,}/{total_rows:,} "
            f"({row_num / total_rows * 100:.1f}%) | "
            f"Pairs: {len(raw_scores):,}"
        )


# ------------------------------------------------------------
# 7. RESULTS
# ------------------------------------------------------------

scores = pd.DataFrame({
    "raw_score": raw_scores,
    "suffix_score": suffix_scores
})

scores["improvement"] = (
    scores["suffix_score"] -
    scores["raw_score"]
)


print("\n" + "=" * 60)
print("GT SUFFIX NORMALIZATION RESULTS")
print("=" * 60)

print(
    f"GT rows processed : {total_rows:,}"
)

print(
    f"GT pairs analyzed : {len(scores):,}"
)


print("\nMean similarity")

print(
    f"Raw    : {scores['raw_score'].mean():.2f}"
)

print(
    f"Suffix : {scores['suffix_score'].mean():.2f}"
)


print("\nMedian similarity")

print(
    f"Raw    : {scores['raw_score'].median():.2f}"
)

print(
    f"Suffix : {scores['suffix_score'].median():.2f}"
)


print("\nChange")

print(
    f"Improved : "
    f"{(scores['improvement'] > 0).mean()*100:.2f}%"
)

print(
    f"Same     : "
    f"{(scores['improvement'] == 0).mean()*100:.2f}%"
)

print(
    f"Worse    : "
    f"{(scores['improvement'] < 0).mean()*100:.2f}%"
)


print("\nSimilarity >= 90")

print(
    f"Raw    : "
    f"{(scores['raw_score'] >= 90).mean()*100:.2f}%"
)

print(
    f"Suffix : "
    f"{(scores['suffix_score'] >= 90).mean()*100:.2f}%"
)


print("\nImprovement statistics")

print(
    scores["improvement"].describe()
)

Creating indexes...
✓ Indexes ready
✓ GT sample: 25,000

Required S1: 25,000
Required S2: 42,060
Required S3: 44,674

Building normalization cache...
Unique names to normalize: 24,143
Normalized 5,000/24,143
Normalized 10,000/24,143
Normalized 15,000/24,143
Normalized 20,000/24,143
✓ Normalization complete

Starting GT comparison...
Progress: 1,000/25,000 (4.0%) | Pairs: 0
Progress: 2,000/25,000 (8.0%) | Pairs: 0
Progress: 3,000/25,000 (12.0%) | Pairs: 0
Progress: 4,000/25,000 (16.0%) | Pairs: 0
Progress: 5,000/25,000 (20.0%) | Pairs: 0
Progress: 6,000/25,000 (24.0%) | Pairs: 0
Progress: 7,000/25,000 (28.0%) | Pairs: 0
Progress: 8,000/25,000 (32.0%) | Pairs: 0
Progress: 9,000/25,000 (36.0%) | Pairs: 0
Progress: 10,000/25,000 (40.0%) | Pairs: 0
Progress: 11,000/25,000 (44.0%) | Pairs: 0
Progress: 12,000/25,000 (48.0%) | Pairs: 0
Progress: 13,000/25,000 (52.0%) | Pairs: 0
Progress: 14,000/25,000 (56.0%) | Pairs: 0
Progress: 15,000/25,000 (60.0%) | Pairs: 0
Progress: 16,000/25,000 (64.0%)

In [72]:
print("S1 entity_id examples:")
print(source1["entity_id"].head().tolist())

print("\nS2 entity_id examples:")
print(source2["entity_id"].head().tolist())

print("\nS3 entity_id examples:")
print(source3["entity_id"].head().tolist())

print("\nGT S1 example:")
print(gt["source1_entity_id"].iloc[0])

print("\nGT matches example:")
print(gt["matched_entity_ids"].iloc[0])

S1 entity_id examples:
['S1-925783039', 'S1-773889195', 'S1-377745466', 'S1-133037285', 'S1-755362802']

S2 entity_id examples:
['S2-166376419', 'S2-764573417', 'S2-639257739', 'S2-163963287', 'S2-49942811']

S3 entity_id examples:
['S3-202863386', 'S3-859268022', 'S3-22467283', 'S3-671162755', 'S3-960981775']

GT S1 example:
S1-965667

GT matches example:
S2-681193310,S2-743505751,S3-775321672,S3-11291185,S3-860443364


In [73]:
import re
import pandas as pd
from rapidfuzz.fuzz import ratio


# ============================================================
# 1. NORMALIZATION
# ============================================================

def basic_norm(x):
    if pd.isna(x):
        return ""
    
    x = str(x).lower()
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    return re.sub(r"\s+", " ", x).strip()


def suffix_norm(x):
    x = basic_norm(x)

    x = re.sub(
        r"\b(private limited|pvt limited|pvt ltd|public limited|"
        r"limited|ltd|llc|incorporated|inc|corporation|corp|llp)\b",
        " ",
        x
    )

    return re.sub(r"\s+", " ", x).strip()


# ============================================================
# 2. DIRECT ENTITY INDEXES
# ============================================================

print("Creating indexes...")

s1_names = source1.set_index("entity_id")["business_name"].to_dict()
s2_names = source2.set_index("entity_id")["business_name"].to_dict()
s3_names = source3.set_index("entity_id")["business_name"].to_dict()

print("✓ Indexes ready")


# ============================================================
# 3. SAMPLE GT
# ============================================================

N = min(25_000, len(gt))

gt_sample = gt[
    ["source1_entity_id", "matched_entity_ids"]
].sample(
    N,
    random_state=42
)

print(f"✓ GT sample: {len(gt_sample):,}")


# ============================================================
# 4. ANALYZE TRUE MATCHES
# ============================================================

raw_scores = []
suffix_scores = []

print("\nStarting comparison...")

for row_num, row in enumerate(
    gt_sample.itertuples(index=False),
    1
):

    s1_id = row.source1_entity_id

    s1_name = s1_names.get(s1_id)

    if s1_name is None:
        continue

    s1_basic = basic_norm(s1_name)
    s1_suffix = suffix_norm(s1_name)


    # GT is comma-separated
    matched_ids = str(
        row.matched_entity_ids
    ).split(",")


    for matched_id in matched_ids:

        matched_id = matched_id.strip()

        if matched_id.startswith("S2-"):
            target_name = s2_names.get(matched_id)

        elif matched_id.startswith("S3-"):
            target_name = s3_names.get(matched_id)

        else:
            continue


        if target_name is None:
            continue


        target_basic = basic_norm(target_name)
        target_suffix = suffix_norm(target_name)


        raw_scores.append(
            ratio(s1_basic, target_basic)
        )

        suffix_scores.append(
            ratio(s1_suffix, target_suffix)
        )


    if row_num % 1000 == 0:
        print(
            f"Progress: {row_num:,}/{len(gt_sample):,} "
            f"({row_num / len(gt_sample) * 100:.1f}%) | "
            f"Pairs: {len(raw_scores):,}"
        )


# ============================================================
# 5. RESULTS
# ============================================================

scores = pd.DataFrame({
    "raw_score": raw_scores,
    "suffix_score": suffix_scores
})

scores["improvement"] = (
    scores["suffix_score"] -
    scores["raw_score"]
)


print("\n" + "=" * 60)
print("SUFFIX NORMALIZATION — GT RESULTS")
print("=" * 60)

print(f"GT rows       : {len(gt_sample):,}")
print(f"GT pairs      : {len(scores):,}")

print("\nMean similarity")
print(f"Raw           : {scores.raw_score.mean():.2f}")
print(f"Suffix        : {scores.suffix_score.mean():.2f}")

print("\nMedian similarity")
print(f"Raw           : {scores.raw_score.median():.2f}")
print(f"Suffix        : {scores.suffix_score.median():.2f}")

print("\nChange")
print(f"Improved      : {(scores.improvement > 0).mean()*100:.2f}%")
print(f"Same          : {(scores.improvement == 0).mean()*100:.2f}%")
print(f"Worse         : {(scores.improvement < 0).mean()*100:.2f}%")

print("\nSimilarity >= 90")
print(f"Raw           : {(scores.raw_score >= 90).mean()*100:.2f}%")
print(f"Suffix        : {(scores.suffix_score >= 90).mean()*100:.2f}%")

Creating indexes...
✓ Indexes ready
✓ GT sample: 25,000

Starting comparison...
Progress: 1,000/25,000 (4.0%) | Pairs: 3,559
Progress: 2,000/25,000 (8.0%) | Pairs: 6,953
Progress: 3,000/25,000 (12.0%) | Pairs: 10,424
Progress: 4,000/25,000 (16.0%) | Pairs: 13,792
Progress: 5,000/25,000 (20.0%) | Pairs: 17,216
Progress: 6,000/25,000 (24.0%) | Pairs: 20,687
Progress: 7,000/25,000 (28.0%) | Pairs: 24,226
Progress: 8,000/25,000 (32.0%) | Pairs: 27,602
Progress: 9,000/25,000 (36.0%) | Pairs: 31,064
Progress: 10,000/25,000 (40.0%) | Pairs: 34,567
Progress: 11,000/25,000 (44.0%) | Pairs: 38,117
Progress: 12,000/25,000 (48.0%) | Pairs: 41,586
Progress: 13,000/25,000 (52.0%) | Pairs: 45,051
Progress: 14,000/25,000 (56.0%) | Pairs: 48,524
Progress: 15,000/25,000 (60.0%) | Pairs: 51,973
Progress: 16,000/25,000 (64.0%) | Pairs: 55,558
Progress: 17,000/25,000 (68.0%) | Pairs: 59,063
Progress: 18,000/25,000 (72.0%) | Pairs: 62,529
Progress: 19,000/25,000 (76.0%) | Pairs: 66,030
Progress: 20,000/25,0

## 5. Business Name Suffix Normalization Analysis

We evaluated the effect of removing common legal business suffixes such as `LLC`, `Inc`, `Ltd`, `Limited`, `Private Limited`, `Pvt Ltd`, `LLP`, and `Corporation`.

The analysis was performed on **25,000 sampled S1 entities** and **86,734 verified S1→S2/S3 ground-truth pairs**.

### Results

| Metric | Raw Name | Suffix-Normalized | Change |
|---|---:|---:|---:|
| Mean similarity | 78.97 | **80.52** | **+1.55** |
| Median similarity | 87.50 | **89.47** | **+1.97** |
| Similarity ≥ 90 | 44.27% | **49.19%** | **+4.92 pp** |
| Improved pairs | — | **24.85%** | — |
| Unchanged pairs | — | **51.77%** | — |
| Worse pairs | — | **23.38%** | — |

### Key Findings

1. Removing legal business suffixes improves name similarity overall.
2. Mean similarity increases from **78.97 → 80.52**.
3. Median similarity increases from **87.50 → 89.47**.
4. The proportion of high-confidence name matches (similarity ≥ 90) increases from **44.27% → 49.19%**.
5. However, **23.38% of pairs become worse**, showing that suffix removal can also remove useful information.
6. Therefore, suffix normalization should **not replace the original business name**.

### Modeling Decision

We will retain both representations:

- **Raw/standard normalized business name**
- **Suffix-normalized business name**

Both can be used as separate features during entity matching.

This allows the model to learn when legal suffixes are useful and when they should be ignored.

### EDA Conclusion

> **Business-name suffix normalization provides a meaningful matching signal, but it is not universally beneficial. Therefore, both raw and suffix-normalized representations should be retained for downstream candidate matching and model features.**
"""



In [74]:
import re
import pandas as pd
from rapidfuzz.fuzz import ratio


# ============================================================
# ADDRESS NORMALIZATION
# ============================================================

def address_norm(x):
    if pd.isna(x):
        return ""

    x = str(x).lower()

    # Standardize common address abbreviations
    replacements = {
        r"\bstreet\b": "st",
        r"\broad\b": "rd",
        r"\bavenue\b": "ave",
        r"\bdrive\b": "dr",
        r"\blane\b": "ln",
        r"\bparkway\b": "pkwy",
        r"\bhighway\b": "hwy",
        r"\bboulevard\b": "blvd",
        r"\bsuite\b": "ste",
        r"\bapartment\b": "apt",
    }

    for pattern, replacement in replacements.items():
        x = re.sub(pattern, replacement, x)

    # Remove punctuation
    x = re.sub(r"[^a-z0-9\s]", " ", x)

    # Collapse whitespace
    return re.sub(r"\s+", " ", x).strip()


# ============================================================
# INDEXES
# ============================================================

print("Creating indexes...")

s1_addr = source1.set_index("entity_id")["business_address"].to_dict()
s2_addr = source2.set_index("entity_id")["business_address"].to_dict()
s3_addr = source3.set_index("entity_id")["business_address"].to_dict()

print("✓ Indexes ready")


# ============================================================
# GT SAMPLE
# ============================================================

N = min(25_000, len(gt))

gt_sample = gt[
    ["source1_entity_id", "matched_entity_ids"]
].sample(
    N,
    random_state=42
)

print(f"✓ GT sample: {len(gt_sample):,}")


# ============================================================
# MATCH + SCORE
# ============================================================

raw_scores = []
normalized_scores = []

print("\nStarting address comparison...")

for row_num, row in enumerate(
    gt_sample.itertuples(index=False),
    1
):

    s1_id = row.source1_entity_id
    s1_address = s1_addr.get(s1_id)

    if s1_address is None:
        continue

    s1_raw = str(s1_address).lower().strip()
    s1_norm = address_norm(s1_address)

    matched_ids = str(
        row.matched_entity_ids
    ).split(",")

    for matched_id in matched_ids:

        matched_id = matched_id.strip()

        if matched_id.startswith("S2-"):
            target_address = s2_addr.get(matched_id)

        elif matched_id.startswith("S3-"):
            target_address = s3_addr.get(matched_id)

        else:
            continue

        if target_address is None:
            continue

        target_raw = str(target_address).lower().strip()
        target_norm = address_norm(target_address)

        raw_scores.append(
            ratio(s1_raw, target_raw)
        )

        normalized_scores.append(
            ratio(s1_norm, target_norm)
        )

    if row_num % 1000 == 0:
        print(
            f"Progress: {row_num:,}/{len(gt_sample):,} "
            f"({row_num / len(gt_sample) * 100:.1f}%) | "
            f"Pairs: {len(raw_scores):,}"
        )


# ============================================================
# RESULTS
# ============================================================

scores = pd.DataFrame({
    "raw_score": raw_scores,
    "normalized_score": normalized_scores
})

scores["improvement"] = (
    scores["normalized_score"] -
    scores["raw_score"]
)


print("\n" + "=" * 60)
print("BUSINESS ADDRESS NORMALIZATION — GT RESULTS")
print("=" * 60)

print(f"GT rows       : {len(gt_sample):,}")
print(f"GT pairs      : {len(scores):,}")

print("\nMean similarity")
print(f"Raw           : {scores.raw_score.mean():.2f}")
print(f"Normalized    : {scores.normalized_score.mean():.2f}")

print("\nMedian similarity")
print(f"Raw           : {scores.raw_score.median():.2f}")
print(f"Normalized    : {scores.normalized_score.median():.2f}")

print("\nChange")
print(
    f"Improved      : "
    f"{(scores.improvement > 0).mean()*100:.2f}%"
)

print(
    f"Same          : "
    f"{(scores.improvement == 0).mean()*100:.2f}%"
)

print(
    f"Worse         : "
    f"{(scores.improvement < 0).mean()*100:.2f}%"
)

print("\nSimilarity >= 90")
print(
    f"Raw           : "
    f"{(scores.raw_score >= 90).mean()*100:.2f}%"
)

print(
    f"Normalized    : "
    f"{(scores.normalized_score >= 90).mean()*100:.2f}%"
)

print("\nImprovement statistics")
print(scores["improvement"].describe())

Creating indexes...
✓ Indexes ready
✓ GT sample: 25,000

Starting address comparison...
Progress: 1,000/25,000 (4.0%) | Pairs: 3,559
Progress: 2,000/25,000 (8.0%) | Pairs: 6,953
Progress: 3,000/25,000 (12.0%) | Pairs: 10,424
Progress: 4,000/25,000 (16.0%) | Pairs: 13,792
Progress: 5,000/25,000 (20.0%) | Pairs: 17,216
Progress: 6,000/25,000 (24.0%) | Pairs: 20,687
Progress: 7,000/25,000 (28.0%) | Pairs: 24,226
Progress: 8,000/25,000 (32.0%) | Pairs: 27,602
Progress: 9,000/25,000 (36.0%) | Pairs: 31,064
Progress: 10,000/25,000 (40.0%) | Pairs: 34,567
Progress: 11,000/25,000 (44.0%) | Pairs: 38,117
Progress: 12,000/25,000 (48.0%) | Pairs: 41,586
Progress: 13,000/25,000 (52.0%) | Pairs: 45,051
Progress: 14,000/25,000 (56.0%) | Pairs: 48,524
Progress: 15,000/25,000 (60.0%) | Pairs: 51,973
Progress: 16,000/25,000 (64.0%) | Pairs: 55,558
Progress: 17,000/25,000 (68.0%) | Pairs: 59,063
Progress: 18,000/25,000 (72.0%) | Pairs: 62,529
Progress: 19,000/25,000 (76.0%) | Pairs: 66,030
Progress: 20,

# country signal

In [75]:
# ============================================================
# COUNTRY SIGNAL ANALYSIS
# ============================================================

import pandas as pd


# ------------------------------------------------------------
# 1. COUNTRY INDEXES
# ------------------------------------------------------------

print("Creating country indexes...")

s1_country = source1.set_index("entity_id")["country"].to_dict()
s2_country = source2.set_index("entity_id")["country"].to_dict()
s3_country = source3.set_index("entity_id")["country"].to_dict()

print("✓ Indexes ready")


# ------------------------------------------------------------
# 2. SAMPLE GT
# ------------------------------------------------------------

N = min(25_000, len(gt))

gt_sample = gt[
    ["source1_entity_id", "matched_entity_ids"]
].sample(
    N,
    random_state=42
)

print(f"✓ GT sample: {len(gt_sample):,}")


# ------------------------------------------------------------
# 3. COUNTRY COMPARISON
# ------------------------------------------------------------

same_country = 0
different_country = 0
missing_country = 0
total_pairs = 0

country_pairs = {}

print("\nStarting country analysis...")

for row_num, row in enumerate(
    gt_sample.itertuples(index=False),
    1
):

    s1_id = row.source1_entity_id
    c1 = s1_country.get(s1_id)

    if pd.isna(c1):
        c1 = None

    if c1 is not None:
        c1 = str(c1).strip().lower()


    matched_ids = str(
        row.matched_entity_ids
    ).split(",")


    for matched_id in matched_ids:

        matched_id = matched_id.strip()

        if matched_id.startswith("S2-"):
            c2 = s2_country.get(matched_id)

        elif matched_id.startswith("S3-"):
            c2 = s3_country.get(matched_id)

        else:
            continue


        total_pairs += 1

        if pd.isna(c2) or c2 is None:

            missing_country += 1
            continue


        c2 = str(c2).strip().lower()


        if c1 is None or c1 == "":
            missing_country += 1

        elif c1 == c2:
            same_country += 1

        else:
            different_country += 1


        pair = (c1, c2)

        country_pairs[pair] = (
            country_pairs.get(pair, 0) + 1
        )


    if row_num % 1000 == 0:

        print(
            f"Progress: {row_num:,}/{len(gt_sample):,} "
            f"({row_num / len(gt_sample) * 100:.1f}%) | "
            f"Pairs: {total_pairs:,}"
        )


# ------------------------------------------------------------
# 4. RESULTS
# ------------------------------------------------------------

valid_country_pairs = (
    same_country +
    different_country
)

print("\n" + "=" * 60)
print("COUNTRY SIGNAL — GT RESULTS")
print("=" * 60)

print(f"GT rows              : {len(gt_sample):,}")
print(f"GT pairs             : {total_pairs:,}")

print("\nCountry agreement")

print(
    f"Same country        : {same_country:,} "
    f"({same_country / valid_country_pairs * 100:.2f}%)"
)

print(
    f"Different country   : {different_country:,} "
    f"({different_country / valid_country_pairs * 100:.2f}%)"
)

print(
    f"Missing country     : {missing_country:,} "
    f"({missing_country / total_pairs * 100:.2f}%)"
)


# ------------------------------------------------------------
# 5. TOP COUNTRY PAIRS
# ------------------------------------------------------------

pairs_df = pd.DataFrame(
    [
        (c1, c2, count)
        for (c1, c2), count in country_pairs.items()
    ],
    columns=[
        "source1_country",
        "matched_country",
        "count"
    ]
)

pairs_df = pairs_df.sort_values(
    "count",
    ascending=False
)

print("\nTop country pairs:")
display(pairs_df.head(20))

Creating country indexes...
✓ Indexes ready
✓ GT sample: 25,000

Starting country analysis...
Progress: 1,000/25,000 (4.0%) | Pairs: 3,559
Progress: 2,000/25,000 (8.0%) | Pairs: 6,953
Progress: 3,000/25,000 (12.0%) | Pairs: 10,424
Progress: 4,000/25,000 (16.0%) | Pairs: 13,792
Progress: 5,000/25,000 (20.0%) | Pairs: 17,216
Progress: 6,000/25,000 (24.0%) | Pairs: 20,687
Progress: 7,000/25,000 (28.0%) | Pairs: 24,226
Progress: 8,000/25,000 (32.0%) | Pairs: 27,602
Progress: 9,000/25,000 (36.0%) | Pairs: 31,064
Progress: 10,000/25,000 (40.0%) | Pairs: 34,567
Progress: 11,000/25,000 (44.0%) | Pairs: 38,117
Progress: 12,000/25,000 (48.0%) | Pairs: 41,586
Progress: 13,000/25,000 (52.0%) | Pairs: 45,051
Progress: 14,000/25,000 (56.0%) | Pairs: 48,524
Progress: 15,000/25,000 (60.0%) | Pairs: 51,973
Progress: 16,000/25,000 (64.0%) | Pairs: 55,558
Progress: 17,000/25,000 (68.0%) | Pairs: 59,063
Progress: 18,000/25,000 (72.0%) | Pairs: 62,529
Progress: 19,000/25,000 (76.0%) | Pairs: 66,030
Progres

,source1_country,matched_country,count
0,us,us,51923
1,india,india,34811


In [76]:
# ============================================================
# GT MATCH CARDINALITY ANALYSIS
# ============================================================

import pandas as pd

print("Analyzing GT cardinality...")

# Work on a copy
gt_card = gt[[
    "source1_entity_id",
    "matched_entity_ids"
]].copy()

# Count matched entities
gt_card["match_count"] = (
    gt_card["matched_entity_ids"]
    .fillna("")
    .astype(str)
    .apply(
        lambda x: 0 if not x.strip()
        else len([i for i in x.split(",") if i.strip()])
    )
)

# ------------------------------------------------------------
# BASIC STATISTICS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("GT MATCH CARDINALITY")
print("=" * 60)

print(f"Total S1 entities : {len(gt_card):,}")

print("\nMatch count statistics:")
print(gt_card["match_count"].describe())


# ------------------------------------------------------------
# DISTRIBUTION
# ------------------------------------------------------------

distribution = (
    gt_card["match_count"]
    .value_counts()
    .sort_index()
    .reset_index()
)

distribution.columns = [
    "matched_entities",
    "s1_entity_count"
]

distribution["percentage"] = (
    distribution["s1_entity_count"]
    / len(gt_card)
    * 100
)

print("\nMatch-count distribution:")
display(distribution.head(30))


# ------------------------------------------------------------
# CARDINALITY CATEGORIES
# ------------------------------------------------------------

zero = (gt_card["match_count"] == 0).sum()
one = (gt_card["match_count"] == 1).sum()
multiple = (gt_card["match_count"] > 1).sum()

print("\nCardinality categories:")

print(
    f"0 matches       : {zero:,} "
    f"({zero / len(gt_card) * 100:.2f}%)"
)

print(
    f"1 match         : {one:,} "
    f"({one / len(gt_card) * 100:.2f}%)"
)

print(
    f">1 matches      : {multiple:,} "
    f"({multiple / len(gt_card) * 100:.2f}%)"
)


# ------------------------------------------------------------
# S2 VS S3 CONTRIBUTION
# ------------------------------------------------------------

def count_source_ids(x, prefix):
    if pd.isna(x):
        return 0

    return sum(
        item.strip().startswith(prefix)
        for item in str(x).split(",")
    )


gt_card["s2_count"] = gt_card[
    "matched_entity_ids"
].apply(
    lambda x: count_source_ids(x, "S2-")
)

gt_card["s3_count"] = gt_card[
    "matched_entity_ids"
].apply(
    lambda x: count_source_ids(x, "S3-")
)


print("\nS2/S3 match statistics:")

print(
    f"Total S2 links : {gt_card['s2_count'].sum():,}"
)

print(
    f"Total S3 links : {gt_card['s3_count'].sum():,}"
)

print(
    f"S1 entities with S2 match : "
    f"{(gt_card['s2_count'] > 0).sum():,}"
)

print(
    f"S1 entities with S3 match : "
    f"{(gt_card['s3_count'] > 0).sum():,}"
)


# ------------------------------------------------------------
# EXTREME CASES
# ------------------------------------------------------------

print("\nHighest-cardinality S1 entities:")

display(
    gt_card.sort_values(
        "match_count",
        ascending=False
    ).head(20)
)

Analyzing GT cardinality...

GT MATCH CARDINALITY
Total S1 entities : 2,206,821

Match count statistics:
count    2.206821e+06
mean     3.461253e+00
std      1.705323e+00
min      0.000000e+00
25%      2.000000e+00
50%      3.000000e+00
75%      5.000000e+00
max      1.100000e+01
Name: match_count, dtype: float64

Match-count distribution:


,matched_entities,s1_entity_count,percentage
0,0,123247,5.584821
1,1,119157,5.399486
2,2,375212,17.002376
3,3,530841,24.054556
4,4,484115,21.937212
5,5,321957,14.589176
6,6,164868,7.470837
7,7,63968,2.898649
8,8,18680,0.846466
9,9,4205,0.190546



Cardinality categories:
0 matches       : 123,247 (5.58%)
1 match         : 119,157 (5.40%)
>1 matches      : 1,964,417 (89.02%)

S2/S3 match statistics:
Total S2 links : 3,693,619
Total S3 links : 3,944,746
S1 entities with S2 match : 1,919,076
S1 entities with S3 match : 1,940,545

Highest-cardinality S1 entities:


,source1_entity_id,matched_entity_ids,match_count,s2_count,s3_count
2097835,S1-83149500,"S2-900789342,S2-853474906,S2-882475241,S2-3970...",11,5,6
1258930,S1-652990687,"S2-293123116,S2-49738963,S2-427741161,S2-79896...",11,5,6
28954,S1-611116729,"S2-941369831,S2-771297374,S2-496136040,S2-6625...",11,5,6
1408152,S1-667470626,"S2-244496472,S2-140798239,S2-761090325,S2-8474...",11,5,6
726055,S1-745776042,"S2-173308731,S2-848441457,S2-893672651,S2-3528...",11,5,6
1641985,S1-393972796,"S2-56160411,S2-73297215,S2-634531787,S2-245860...",11,5,6
533152,S1-377963262,"S2-428826375,S2-992296923,S2-616272995,S2-1190...",11,5,6
1709310,S1-873066411,"S2-308890860,S2-187058817,S2-972165829,S2-7729...",11,5,6
2158257,S1-439823948,"S2-473599419,S2-9772066,S2-900794900,S2-131769...",11,5,6
315443,S1-806895726,"S2-13215685,S2-894126623,S2-861293919,S2-27143...",11,5,6


In [77]:
# ============================================================
# S2 vs S3 MATCH CONTRIBUTION ANALYSIS
# ============================================================

import pandas as pd
import numpy as np

print("Analyzing S2 vs S3 contribution...")

df = gt[[
    "source1_entity_id",
    "matched_entity_ids"
]].copy()


# ------------------------------------------------------------
# COUNT S2 / S3 MATCHES
# ------------------------------------------------------------

def count_prefix(x, prefix):
    if pd.isna(x):
        return 0

    return sum(
        item.strip().startswith(prefix)
        for item in str(x).split(",")
        if item.strip()
    )


df["s2_count"] = df["matched_entity_ids"].map(
    lambda x: count_prefix(x, "S2-")
)

df["s3_count"] = df["matched_entity_ids"].map(
    lambda x: count_prefix(x, "S3-")
)

df["total_count"] = (
    df["s2_count"] +
    df["s3_count"]
)


# ============================================================
# OVERALL STATISTICS
# ============================================================

print("\n" + "=" * 60)
print("S2 vs S3 MATCH CONTRIBUTION")
print("=" * 60)

print(f"S1 entities: {len(df):,}")

print("\nTotal links:")
print(f"S2 links : {df['s2_count'].sum():,}")
print(f"S3 links : {df['s3_count'].sum():,}")
print(f"Total    : {df['total_count'].sum():,}")


total_links = df["total_count"].sum()

print("\nLink share:")
print(
    f"S2 : {df['s2_count'].sum() / total_links * 100:.2f}%"
)

print(
    f"S3 : {df['s3_count'].sum() / total_links * 100:.2f}%"
)


# ============================================================
# S1 COVERAGE
# ============================================================

s2_only = (
    (df["s2_count"] > 0) &
    (df["s3_count"] == 0)
).sum()

s3_only = (
    (df["s3_count"] > 0) &
    (df["s2_count"] == 0)
).sum()

both = (
    (df["s2_count"] > 0) &
    (df["s3_count"] > 0)
).sum()

neither = (
    (df["s2_count"] == 0) &
    (df["s3_count"] == 0)
).sum()


print("\nS1 coverage:")

print(
    f"S2 only : {s2_only:,} "
    f"({s2_only / len(df) * 100:.2f}%)"
)

print(
    f"S3 only : {s3_only:,} "
    f"({s3_only / len(df) * 100:.2f}%)"
)

print(
    f"Both    : {both:,} "
    f"({both / len(df) * 100:.2f}%)"
)

print(
    f"Neither : {neither:,} "
    f"({neither / len(df) * 100:.2f}%)"
)


# ============================================================
# DISTRIBUTION
# ============================================================

print("\nAverage links per S1:")

print(
    f"S2 mean : {df['s2_count'].mean():.2f}"
)

print(
    f"S3 mean : {df['s3_count'].mean():.2f}"
)

print(
    f"Total mean : {df['total_count'].mean():.2f}"
)


# ============================================================
# S2/S3 BALANCE
# ============================================================

df["s2_share"] = np.where(
    df["total_count"] > 0,
    df["s2_count"] / df["total_count"],
    np.nan
)

print("\nS2 share among matched S1 entities:")
print(df["s2_share"].describe())


# ============================================================
# DISTRIBUTION OF S2/S3 COUNTS
# ============================================================

print("\nS2 count distribution:")
display(
    df["s2_count"]
    .value_counts()
    .sort_index()
    .head(20)
)

print("\nS3 count distribution:")
display(
    df["s3_count"]
    .value_counts()
    .sort_index()
    .head(20)
)

Analyzing S2 vs S3 contribution...

S2 vs S3 MATCH CONTRIBUTION
S1 entities: 2,206,821

Total links:
S2 links : 3,693,619
S3 links : 3,944,746
Total    : 7,638,365

Link share:
S2 : 48.36%
S3 : 51.64%

S1 coverage:
S2 only : 143,029 (6.48%)
S3 only : 164,498 (7.45%)
Both    : 1,776,047 (80.48%)
Neither : 123,247 (5.58%)

Average links per S1:
S2 mean : 1.67
S3 mean : 1.79
Total mean : 3.46

S2 share among matched S1 entities:
count    2.083574e+06
mean     4.847688e-01
std      2.424387e-01
min      0.000000e+00
25%      3.333333e-01
50%      5.000000e-01
75%      6.666667e-01
max      1.000000e+00
Name: s2_share, dtype: float64

S2 count distribution:


s2_count
0    287745
1    789108
2    652779
3    333957
4    119078
5     24154
Name: count, dtype: int64


S3 count distribution:


s3_count
0    266276
1    716417
2    668375
3    372443
4    145116
5     35378
6      2816
Name: count, dtype: int64

In [78]:
# ============================================================
# DUPLICATE BUSINESS-NAME ANALYSIS
# ============================================================

import pandas as pd

def analyze_duplicates(df, source_name):
    print("\n" + "=" * 60)
    print(f"{source_name} — BUSINESS NAME DUPLICATES")
    print("=" * 60)

    names = df["business_name"].fillna("").astype(str).str.strip()

    counts = names.value_counts()

    duplicate_rows = (names.map(counts) > 1).sum()
    unique_names = (counts == 1).sum()
    duplicated_names = (counts > 1).sum()

    print(f"Total rows              : {len(df):,}")
    print(f"Unique names             : {len(counts):,}")
    print(f"Names occurring once     : {unique_names:,}")
    print(f"Names occurring >1       : {duplicated_names:,}")
    print(f"Rows with duplicate name : {duplicate_rows:,}")
    print(
        f"Duplicate-row percentage : "
        f"{duplicate_rows / len(df) * 100:.2f}%"
    )

    print("\nDuplicate frequency distribution:")
    print(counts[counts > 1].value_counts().sort_index().head(20))

    print("\nMost frequent business names:")
    display(
        counts[counts > 1]
        .head(20)
        .rename("count")
        .reset_index()
        .rename(columns={"index": "business_name"})
    )

    return counts


s1_name_counts = analyze_duplicates(source1, "S1")
s2_name_counts = analyze_duplicates(source2, "S2")
s3_name_counts = analyze_duplicates(source3, "S3")


S1 — BUSINESS NAME DUPLICATES
Total rows              : 2,206,821
Unique names             : 1,539,229
Names occurring once     : 1,361,436
Names occurring >1       : 177,793
Rows with duplicate name : 845,385
Duplicate-row percentage : 38.31%

Duplicate frequency distribution:
count
2     99182
3     31299
4     14056
5      7898
6      5126
7      3703
8      2696
9      1863
10     1272
11      852
12      553
13      387
14      249
15      151
16      125
17       90
18       78
19       76
20      102
21      118
Name: count, dtype: int64

Most frequent business names:


,business_name,count
0,Primary Care Group,253
1,Ear Nose & Throat Group,251
2,Pediatric Group,222
3,Womens Health Group,220
4,Physical Therapy Group,218
5,Pediatric Dental Group,216
6,Behavioral Health Group,215
7,Chiropractic Group,209
8,Orthopedic Group,208
9,Eye Group,207



S2 — BUSINESS NAME DUPLICATES
Total rows              : 5,034,616
Unique names             : 4,402,009
Names occurring once     : 4,162,230
Names occurring >1       : 239,779
Rows with duplicate name : 872,386
Duplicate-row percentage : 17.33%

Duplicate frequency distribution:
count
2     145547
3      37694
4      18033
5      10499
6       6691
7       4425
8       3006
9       2046
10      1568
11      1151
12       956
13       751
14       644
15       578
16       507
17       440
18       399
19       428
20       377
21       380
Name: count, dtype: int64

Most frequent business names:


,business_name,count
0,Primary Care,320
1,Physical Therapy,307
2,Urgent Care,297
3,Womens Health,297
4,Behavioral Health,296
5,Internal Medicine,288
6,Pediatric Dentistry,274
7,earnosethroat.com,269
8,Pediatric Dental,268
9,Ear Nose & Throat,220



S3 — BUSINESS NAME DUPLICATES
Total rows              : 5,285,603
Unique names             : 4,651,609
Names occurring once     : 4,393,333
Names occurring >1       : 258,276
Rows with duplicate name : 892,270
Duplicate-row percentage : 16.88%

Duplicate frequency distribution:
count
2     164347
3      41199
4      18382
5       9720
6       5652
7       3549
8       2302
9       1672
10      1354
11      1134
12      1074
13       878
14       815
15       700
16       635
17       516
18       427
19       433
20       399
21       365
Name: count, dtype: int64

Most frequent business names:


,business_name,count
0,Primary Care,421
1,Physical Therapy,399
2,Pediatric Dental,393
3,Womens Health,379
4,Urgent Care,377
5,Pediatric Dentistry,361
6,Behavioral Health,330
7,Pediatric,321
8,Internal Medicine,319
9,The Dent,254


In [79]:
# ============================================================
# COUNTRY + BUSINESS NAME UNIQUENESS
# ============================================================

def country_name_analysis(df, source_name):

    print("\n" + "=" * 60)
    print(f"{source_name} — COUNTRY + NAME ANALYSIS")
    print("=" * 60)

    tmp = df[["country", "business_name"]].copy()

    tmp["country"] = (
        tmp["country"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    tmp["business_name"] = (
        tmp["business_name"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    # Count occurrences of each country + name combination
    counts = (
        tmp.groupby(
            ["country", "business_name"],
            sort=False
        )
        .size()
    )

    # Rows belonging to unique country+name combinations
    unique_combo_rows = (
        counts[counts == 1].sum()
    )

    # Rows belonging to duplicated country+name combinations
    duplicate_combo_rows = (
        counts[counts > 1].sum()
    )

    total = len(tmp)

    print(f"Total rows                    : {total:,}")
    print(f"Unique country+name combos    : {len(counts):,}")
    print(
        f"Rows uniquely identified      : "
        f"{unique_combo_rows:,} "
        f"({unique_combo_rows / total * 100:.2f}%)"
    )
    print(
        f"Rows still duplicated         : "
        f"{duplicate_combo_rows:,} "
        f"({duplicate_combo_rows / total * 100:.2f}%)"
    )

    # Distribution of duplicate group sizes
    print("\nCountry+name group-size distribution:")

    distribution = (
        counts.value_counts()
        .sort_index()
        .head(20)
        .rename_axis("group_size")
        .reset_index(name="combination_count")
    )

    display(distribution)

    # Largest ambiguous groups
    print("\nLargest country+name groups:")

    largest = (
        counts
        .sort_values(ascending=False)
        .head(20)
        .rename("count")
        .reset_index()
    )

    display(largest)

    return counts


s1_country_name = country_name_analysis(
    source1,
    "S1"
)

s2_country_name = country_name_analysis(
    source2,
    "S2"
)

s3_country_name = country_name_analysis(
    source3,
    "S3"
)


S1 — COUNTRY + NAME ANALYSIS
Total rows                    : 2,206,821
Unique country+name combos    : 1,539,470
Rows uniquely identified      : 1,361,407 (61.69%)
Rows still duplicated         : 845,414 (38.31%)

Country+name group-size distribution:


,group_size,combination_count
0,1,1361407
1,2,99410
2,3,31377
3,4,14083
4,5,7894
5,6,5117
6,7,3713
7,8,2677
8,9,1858
9,10,1260



Largest country+name groups:


,country,business_name,count
0,us,primary care group,253
1,us,ear nose & throat group,251
2,us,pediatric group,222
3,us,womens health group,220
4,us,physical therapy group,218
5,us,pediatric dental group,216
6,us,behavioral health group,215
7,us,chiropractic group,209
8,us,orthopedic group,208
9,us,eye group,207



S2 — COUNTRY + NAME ANALYSIS
Total rows                    : 5,034,616
Unique country+name combos    : 4,291,875
Rows uniquely identified      : 4,005,443 (79.56%)
Rows still duplicated         : 1,029,173 (20.44%)

Country+name group-size distribution:


...

,country,business_name,count
0,us,primary care,397
1,us,physical therapy,386
2,us,womens health,369
3,us,urgent care,366
4,us,behavioral health,363
5,us,earnosethroat.com,348
6,us,internal medicine,346
7,us,pediatric dental,337
8,us,pediatric dentistry,335
9,us,ear nose & throat,253



S3 — COUNTRY + NAME ANALYSIS
Total rows                    : 5,285,603
Unique country+name combos    : 4,585,887
Rows uniquely identified      : 4,300,228 (81.36%)
Rows still duplicated         : 985,375 (18.64%)

Country+name group-size distribution:


,group_size,combination_count
0,1,4300228
1,2,180926
2,3,44795
3,4,20037
4,5,10895
5,6,6534
6,7,4226
7,8,2838
8,9,2266
9,10,1874



Largest country+name groups:


,country,business_name,count
0,us,primary care,421
1,us,physical therapy,399
2,us,pediatric dental,393
3,us,womens health,379
4,us,urgent care,377
5,us,pediatric dentistry,361
6,us,behavioral health,330
7,us,pediatric,321
8,us,internal medicine,319
9,us,earnosethroat.com,307


In [80]:
# ============================================================
# COUNTRY + NORMALIZED NAME + ADDRESS UNIQUENESS
# ============================================================

import re
import pandas as pd


def normalize_name(x):
    if pd.isna(x):
        return ""
    x = str(x).lower()
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    return re.sub(r"\s+", " ", x).strip()


def normalize_address(x):
    if pd.isna(x):
        return ""

    x = str(x).lower()

    replacements = {
        r"\bstreet\b": "st",
        r"\broad\b": "rd",
        r"\bavenue\b": "ave",
        r"\bdrive\b": "dr",
        r"\blane\b": "ln",
        r"\bparkway\b": "pkwy",
        r"\bhighway\b": "hwy",
        r"\bboulevard\b": "blvd",
        r"\bsuite\b": "ste",
        r"\bapartment\b": "apt",
    }

    for pattern, replacement in replacements.items():
        x = re.sub(pattern, replacement, x)

    x = re.sub(r"[^a-z0-9\s]", " ", x)
    return re.sub(r"\s+", " ", x).strip()


def analyze_composite_uniqueness(df, source_name):

    print("\n" + "=" * 60)
    print(f"{source_name} — COUNTRY + NAME + ADDRESS")
    print("=" * 60)

    tmp = df[
        ["country", "business_name", "business_address"]
    ].copy()

    # Normalize country
    tmp["country"] = (
        tmp["country"]
        .fillna("")
        .astype(str)
        .str.lower()
        .str.strip()
    )

    # Normalize name
    print("Normalizing names...")
    tmp["norm_name"] = (
        tmp["business_name"]
        .map(normalize_name)
    )

    # Normalize address
    print("Normalizing addresses...")
    tmp["norm_address"] = (
        tmp["business_address"]
        .map(normalize_address)
    )

    print("Creating composite groups...")

    counts = (
        tmp.groupby(
            [
                "country",
                "norm_name",
                "norm_address"
            ],
            sort=False
        )
        .size()
    )

    total = len(tmp)

    unique_rows = counts[counts == 1].sum()
    duplicate_rows = counts[counts > 1].sum()

    print(f"\nTotal rows                 : {total:,}")
    print(f"Unique composite groups   : {len(counts):,}")

    print(
        f"Uniquely identified       : "
        f"{unique_rows:,} "
        f"({unique_rows / total * 100:.2f}%)"
    )

    print(
        f"Still duplicated          : "
        f"{duplicate_rows:,} "
        f"({duplicate_rows / total * 100:.2f}%)"
    )

    # Duplicate group distribution
    print("\nComposite group-size distribution:")

    distribution = (
        counts.value_counts()
        .sort_index()
        .head(20)
        .rename_axis("group_size")
        .reset_index(name="combination_count")
    )

    display(distribution)

    # Largest remaining duplicates
    print("\nLargest remaining composite groups:")

    largest = (
        counts[counts > 1]
        .sort_values(ascending=False)
        .head(20)
        .rename("count")
        .reset_index()
    )

    display(largest)

    return counts


# ============================================================
# RUN
# ============================================================

s1_composite = analyze_composite_uniqueness(
    source1,
    "S1"
)

s2_composite = analyze_composite_uniqueness(
    source2,
    "S2"
)

s3_composite = analyze_composite_uniqueness(
    source3,
    "S3"
)


S1 — COUNTRY + NAME + ADDRESS
Normalizing names...
Normalizing addresses...
Creating composite groups...

Total rows                 : 2,206,821
Unique composite groups   : 2,206,821
Uniquely identified       : 2,206,821 (100.00%)
Still duplicated          : 0 (0.00%)

Composite group-size distribution:


,group_size,combination_count
0,1,2206821



Largest remaining composite groups:


,country,norm_name,norm_address,count



S2 — COUNTRY + NAME + ADDRESS
Normalizing names...
Normalizing addresses...
Creating composite groups...

Total rows                 : 5,034,616
Unique composite groups   : 4,938,748
Uniquely identified       : 4,849,064 (96.31%)
Still duplicated          : 185,552 (3.69%)

Composite group-size distribution:


...

,country,norm_name,norm_address,count
0,india,,2 mumbai maharashtra,10
1,india,,501 mumbai mumbai city maharashtra,10
2,india,,floor mumbai maharashtra,10
3,india,,8 kolkata howrah west bengal,9
4,india,,6 mumbai mumbai city maharashtra,8
5,india,,1 kolkata howrah west bengal,8
6,india,,no 4 chennai tamil nadu,8
7,india,,ground floor bangalore karnataka,8
8,india,,no 6 bangalore karnataka,8
9,india,,no 20 chennai tamil nadu,8



S3 — COUNTRY + NAME + ADDRESS
Normalizing names...
Normalizing addresses...
Creating composite groups...

Total rows                 : 5,285,603
Unique composite groups   : 5,211,377
Uniquely identified       : 5,141,530 (97.27%)
Still duplicated          : 144,073 (2.73%)

Composite group-size distribution:


,group_size,combination_count
0,1,5141530
1,2,65808
2,3,3774
3,4,217
4,5,27
5,6,16
6,7,4
7,8,1



Largest remaining composite groups:


,country,norm_name,norm_address,count
0,india,,5 ahmedabad gj,8
1,india,,no 2 chennai tn,7
2,india,,2 kolkata howrah wb,7
3,india,,no 7 bangalore ka,7
4,india,,18 kolkata howrah wb,7
5,india,,5 mumbai mumbai city mh,6
6,india,star,,6
7,india,seven,,6
8,india,swastik,,6
9,india,southern,,6


In [81]:
# ============================================================
# FINAL EDA — EXACT COMPOSITE MATCH COVERAGE
# country + normalized_name + normalized_address
# ============================================================

import re
import pandas as pd


def norm_name(x):
    if pd.isna(x):
        return ""
    x = str(x).lower()
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    return re.sub(r"\s+", " ", x).strip()


def norm_address(x):
    if pd.isna(x):
        return ""
    
    x = str(x).lower()

    replacements = {
        r"\bstreet\b": "st",
        r"\broad\b": "rd",
        r"\bavenue\b": "ave",
        r"\bdrive\b": "dr",
        r"\blane\b": "ln",
        r"\bparkway\b": "pkwy",
        r"\bhighway\b": "hwy",
        r"\bboulevard\b": "blvd",
        r"\bsuite\b": "ste",
        r"\bapartment\b": "apt",
    }

    for pattern, replacement in replacements.items():
        x = re.sub(pattern, replacement, x)

    x = re.sub(r"[^a-z0-9\s]", " ", x)
    return re.sub(r"\s+", " ", x).strip()


# ============================================================
# BUILD COMPOSITE INDEX
# ============================================================

print("Building composite indexes...")

def build_index(df):
    index = {}

    for row in df[
        ["entity_id", "country", "business_name", "business_address"]
    ].itertuples(index=False):

        entity_id, country, name, address = row

        key = (
            str(country).lower().strip(),
            norm_name(name),
            norm_address(address)
        )

        index.setdefault(key, []).append(entity_id)

    return index


s2_index = build_index(source2)
s3_index = build_index(source3)

print("✓ S2 index ready")
print("✓ S3 index ready")


# ============================================================
# SAMPLE GT
# ============================================================

gt_sample = gt[
    ["source1_entity_id", "matched_entity_ids"]
].sample(
    min(25_000, len(gt)),
    random_state=42
)

print(f"✓ GT sample: {len(gt_sample):,}")


# ============================================================
# S1 LOOKUP
# ============================================================

s1_lookup = source1.set_index("entity_id")[
    ["country", "business_name", "business_address"]
].to_dict("index")


# ============================================================
# EVALUATION
# ============================================================

total_gt_pairs = 0
exact_recovered = 0
exact_not_recovered = 0

rows_with_at_least_one_exact = 0

print("\nStarting exact composite evaluation...")

for i, row in enumerate(
    gt_sample.itertuples(index=False),
    1
):

    s1_id = row.source1_entity_id

    s1 = s1_lookup.get(s1_id)

    if s1 is None:
        continue

    key = (
        str(s1["country"]).lower().strip(),
        norm_name(s1["business_name"]),
        norm_address(s1["business_address"])
    )

    candidates = set()

    candidates.update(s2_index.get(key, []))
    candidates.update(s3_index.get(key, []))

    gt_ids = {
        x.strip()
        for x in str(row.matched_entity_ids).split(",")
        if x.strip()
    }

    total_gt_pairs += len(gt_ids)

    intersection = candidates.intersection(gt_ids)

    if intersection:
        rows_with_at_least_one_exact += 1
        exact_recovered += len(intersection)

    exact_not_recovered += (
        len(gt_ids) - len(intersection)
    )

    if i % 1000 == 0:
        print(
            f"Progress: {i:,}/{len(gt_sample):,} "
            f"({i / len(gt_sample) * 100:.1f}%) | "
            f"GT pairs: {total_gt_pairs:,} | "
            f"Recovered: {exact_recovered:,}"
        )


# ============================================================
# FINAL RESULT
# ============================================================

print("\n" + "=" * 60)
print("FINAL EDA — EXACT COMPOSITE MATCH COVERAGE")
print("=" * 60)

print(f"GT rows                    : {len(gt_sample):,}")
print(f"GT pairs                   : {total_gt_pairs:,}")

print(
    f"\nGT pairs recovered exactly : "
    f"{exact_recovered:,}"
)

print(
    f"GT pairs missed            : "
    f"{exact_not_recovered:,}"
)

print(
    f"\nPAIR COVERAGE              : "
    f"{exact_recovered / total_gt_pairs * 100:.2f}%"
)

print(
    f"S1 rows with exact match  : "
    f"{rows_with_at_least_one_exact:,}"
)

print(
    f"ROW COVERAGE               : "
    f"{rows_with_at_least_one_exact / len(gt_sample) * 100:.2f}%"
)

Building composite indexes...
✓ S2 index ready
✓ S3 index ready
✓ GT sample: 25,000

Starting exact composite evaluation...
Progress: 1,000/25,000 (4.0%) | GT pairs: 3,609 | Recovered: 71
Progress: 2,000/25,000 (8.0%) | GT pairs: 7,056 | Recovered: 146
Progress: 3,000/25,000 (12.0%) | GT pairs: 10,577 | Recovered: 218
Progress: 4,000/25,000 (16.0%) | GT pairs: 14,006 | Recovered: 283
Progress: 5,000/25,000 (20.0%) | GT pairs: 17,495 | Recovered: 357
Progress: 6,000/25,000 (24.0%) | GT pairs: 21,026 | Recovered: 440
Progress: 7,000/25,000 (28.0%) | GT pairs: 24,607 | Recovered: 528
Progress: 8,000/25,000 (32.0%) | GT pairs: 28,045 | Recovered: 603
Progress: 9,000/25,000 (36.0%) | GT pairs: 31,561 | Recovered: 686
Progress: 10,000/25,000 (40.0%) | GT pairs: 35,124 | Recovered: 773
Progress: 11,000/25,000 (44.0%) | GT pairs: 38,723 | Recovered: 853
Progress: 12,000/25,000 (48.0%) | GT pairs: 42,250 | Recovered: 930
Progress: 13,000/25,000 (52.0%) | GT pairs: 45,772 | Recovered: 998
Progre